# MedCLIPSeg on Google Colab — v0.3.4

**v0.3.4:** checkpoints are kept on Drive only for C, H1, H2 at the seeds in `KEEP_SEEDS` (default 666 → ~3 GB), to stay well inside a 15 GB Drive shared with Gmail.

**v0.3.3:** `KEEP_CKPT` now actually controls `run()` (it was ignored before); per-experiment `keep` flags in X2's `EXP`; a full Drive gives a warning instead of crashing a finished training.

**v0.3.2:** `train()` logs every training's duration and GPU to `results/train_times.csv`; a **Budget check** cell (after *All analyses*) projects the GPU-hours and compute units still needed from measured times.

**v0.3.1 (TA feedback):** IoU next to Dice everywhere; prompt-variation suite with test-only **synonyms**, **paraphrases** and **word-order** changes (28 designs in groups); H1 now trains with *different* synonyms / word orders so the test variations stay unseen; H1/H2 compared against repo-trained baseline seeds (same pipeline).

**What's new in v0.3 (experiments H0–H3)** — jump to **Section X** for the run plan.
* **Experiment patches** (after the compatibility patches): opt-in Dice-loss fix (`MSC_DICE_FIX=1`), per-sample prompt mixing (`||` in a prompt cell), and raw per-image uncertainty / logit / calibration stats written by `test.py`. Default behaviour is unchanged, so all earlier results stay valid.
* `evaluate()` now merges those raw stats into the per-image CSV and adds `ECE` / `unc_AUROC` to the summary row (raw `.npz` saved to `results/raw/`).
* `train()` takes `env=` (environment variables) and `keep=` (copy this checkpoint to Drive).
* **Section X**: prompt files for every arm, an idempotent `run()` (skips finished trainings/evaluations, restores kept checkpoints), analysis functions with the pre-registered predictions, and the step-by-step run cells.

Runs the official [MedCLIPSeg](https://github.com/HealthX-Lab/MedCLIPSeg) code (CVPR 2026) on Colab, saves every result to Google Drive as it is produced, and adds CPU‑only analysis sections.

**What's new in v0.1**
* **Nothing is lost when Colab recycles the VM:** every `evaluate()` immediately writes its summary row, its per‑image CSV and (optionally) its predicted masks + uncertainty maps to Drive. `save_tables()` writes all tables to one Excel workbook + CSVs.
* **Section R** restores what the first session saved (`results_*.csv`, `csv_logs_*.tgz`).
* **Section A – data understanding** and **Section B – claim checks** run **without a GPU**.
* **Tier 2d – controlled prompt edits:** change one prompt attribute at a time (the repo's prompt "designs" change several at once).

| Section | Needs GPU? | What |
|---|---|---|
| 0 Setup, data | no | env, patches, datasets |
| R Restore | no | import session‑1 results into the new format |
| A Data understanding | no | prompt inventory, attribute consistency across prompt designs, prompt ↔ mask agreement |
| B Claim checks | no (slow‑ish on CPU) | normal images per prompt design, Dice‑loss shape, logit and soft‑target diagnostics |
| T1–T2 | **yes** | released‑checkpoint evaluation, prompt robustness, MC sampling |
| T3 | **yes** | training from scratch |

Drive layout (`DRIVE_DIR/results/`): `all_results.csv` · `per_image/<run>.csv` · `maps/<run>.zip` · `train_logs/` · `tables/medclipseg_tables.xlsx` (+ one CSV per table).

## 0. Setup (works on CPU or GPU runtimes)

In [ ]:
import torch, sys, subprocess
HAS_GPU = torch.cuda.is_available()
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", HAS_GPU)
if HAS_GPU:
    print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
else:
    print("CPU runtime: sections 0, R, A, B work. Tiers T1-T3 need a GPU (Runtime > Change runtime type).")

In [ ]:
# ---- EDIT THESE ----
REPO_URL  = "https://github.com/EhsanSl/MedCLIPSeg.git"
BRANCH    = "main"
USE_DRIVE = True                                    # keep zips + results on Google Drive between sessions
DRIVE_DIR = "/content/drive/MyDrive/MedCLIPSeg_colab"
SEED      = 666                                     # seed used for all released checkpoints
SAVE_MAPS = True                                    # zip predicted masks + uncertainty maps of every evaluation to Drive
KEEP_CKPT = False                                   # True = copy EVERY trained checkpoint (~1 GB each) to Drive. False = only the
                                                    # experiments marked keep=True in X2's EXP (C, H1, H2) at the seeds in KEEP_SEEDS
# --------------------
import os
REPO = "/content/MedCLIPSeg"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/zips", exist_ok=True)
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1

### Install dependencies
`requirements.txt` pins `torch==2.9.0` and `numpy<2.0`. On Colab we **keep the preinstalled torch/torchvision/numpy** (avoids a ~2 GB download and a forced restart). pip will print an `ERROR ... gradio / diffusers requires huggingface-hub>=1...` line: harmless, MedCLIPSeg never imports those packages.

In [ ]:
!grep -v -E "^(torch|torchvision|numpy)([=<>]|$)" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
import importlib
for m in ["torch", "torchvision", "numpy", "transformers", "monai", "timm", "cv2", "huggingface_hub", "pandas"]:
    print(f"{m:16s}", importlib.import_module(m).__version__)

### Compatibility patches (idempotent)
1. `np.Inf → np.inf` (NumPy 2) — otherwise `utils/eval.py` crashes.
2. DataLoader `num_workers=8` → number of Colab CPUs.
3. **Bug fix:** `train.py --resume` looked for the checkpoint in `trained_models/` instead of `trained_models/seed{N}/`, so resuming after a disconnect silently restarted from epoch 0.

These only touch the Colab copy of the repo; `git diff` shows exactly what changed.

In [ ]:
%%writefile /content/colab_patches.py
"""Small compatibility/robustness patches for running MedCLIPSeg on Colab. Idempotent."""
import os, re, pathlib
def _read(path):
    raw = open(path, newline="").read(); crlf = "\r\n" in raw
    return raw.replace("\r\n", "\n"), crlf
def _write(path, s, crlf):
    open(path, "w", newline="").write(s.replace("\n", "\r\n") if crlf else s)
def patch(path, old, new):
    s, crlf = _read(path)
    if new in s: print(f"[skip] {path}: already patched"); return
    assert old in s, f"pattern not found in {path}: {old[:60]!r}"
    _write(path, s.replace(old, new), crlf); print(f"[ok]   {path}")
# 1) NumPy 2.x removed np.Inf (Colab ships NumPy 2) -> eval.py crashes without this
s, crlf = _read("utils/SurfaceDice.py")
_write("utils/SurfaceDice.py", re.sub(r"np\.Inf\b", "np.inf", s), crlf); print("[ok]   utils/SurfaceDice.py (np.Inf -> np.inf)")
# 2) DataLoader workers: repo hard-codes 8; Colab has 2-12 vCPUs
n = min(8, os.cpu_count() or 2)
patch("train.py", "num_workers=8,\n                            pin_memory=True,)", f"num_workers={n},\n                            pin_memory=True,)")
patch("train.py", "num_workers=8,\n                        pin_memory=True)", f"num_workers={n},\n                        pin_memory=True)")
# 3) --resume looked in the wrong folder (missing seed{N}/), so it silently restarted from epoch 0
patch("train.py",
      '            "trained_models",\n            f"{results_name}_latest.pth")',
      '            "trained_models",\n            f"seed{cfg.seed}",\n            f"{results_name}_latest.pth")')
patch("train.py", "checkpoint = torch.load(resume_path)", "checkpoint = torch.load(resume_path, map_location=cfg.MODEL.DEVICE, weights_only=False)")

In [ ]:
!python /content/colab_patches.py
!git diff --stat

### Experiment patches (v0.3) — opt-in, default behaviour unchanged
Run after the compatibility patches, every session. See the docstring for what each patch does.

In [ ]:
%%writefile /content/msc_exp_patches.py
"""Experiment patches for MedCLIPSeg (v0.3). Idempotent. Default behaviour is UNCHANGED:
  1) train.py   - MSC_DICE_FIX=1 (env var): Dice on (B,1,H,W) pooled over the batch, instead of the repo's (B,H,W)
                  call that treats image rows as channels. Batch pooling, because a per-image Dice on the 17%
                  normal (empty-mask) images still rewards background logits below ~-25.                  [H2]
  2) dataloader - a prompt cell containing "||" is a list of alternatives; one is drawn per access    [H1]
  3) test.py    - also writes raw per-image uncertainty / logit / calibration stats next to the masks  [H2, H3]
                  (seg_results/seed{N}/raw_Prompt-{design}.csv + .npz); masks and DSC are unaffected
  4) train.py   - checkpoint I/O: the repo writes a full ~1 GB checkpoint EVERY epoch (+ an unused best-Dice one).
                  Now: latest every MSC_SAVE_EVERY epochs (default 10) and at the last epoch; best-Dice only if
                  MSC_SAVE_BEST=1 (testing uses TEST.USE_LATEST). Training itself is unchanged; resume granularity = 10 epochs.
"""
import re

def _read(path):
    raw = open(path, newline="").read(); crlf = "\r\n" in raw
    return raw.replace("\r\n", "\n"), crlf

def _write(path, s, crlf):
    open(path, "w", newline="").write(s.replace("\n", "\r\n") if crlf else s)

def patch(path, old, new, marker):
    s, crlf = _read(path)
    if marker in s: print(f"[skip] {path}: {marker} already applied"); return
    assert s.count(old) == 1, f"pattern found {s.count(old)}x in {path}: {old[:70]!r}"
    _write(path, s.replace(old, new), crlf); print(f"[ok]   {path}: {marker}")

# ---------------------------------------------------------------- 1) Dice fix (opt-in) ----
patch("train.py",
      "    loss_dice = dice_loss(low_res_logits, low_res_label_batch)\n",
      "    if os.environ.get(\"MSC_DICE_FIX\") == \"1\":   # [msc-dicefix] Dice on (B,1,H,W), pooled over the batch\n"
      "        loss_dice = monai.losses.DiceLoss(sigmoid=True, batch=True)(low_res_logits.unsqueeze(1), low_res_label_batch.unsqueeze(1))\n"
      "    else:                                       # repo behaviour: (B,H,W) -> rows treated as channels\n"
      "        loss_dice = dice_loss(low_res_logits, low_res_label_batch)\n",
      "[msc-dicefix]")
patch("train.py",
      "logger.info(\"Setting fixed seed: {}\".format(cfg.seed))\n",
      "logger.info(\"Setting fixed seed: {}\".format(cfg.seed))\n"
      "    logger.info(f\"[msc] MSC_DICE_FIX={os.environ.get('MSC_DICE_FIX', '0')} prompts={cfg.DATASET.TEXT_PROMPT_PATH}\")\n",
      "[msc] MSC_DICE_FIX=")

# ---------------------------------------------------------------- 2) prompt alternatives ----
patch("datasets/dataloader.py",
      "        image_filename, mask_filename, text = self.data_pairs[idx]\n",
      "        image_filename, mask_filename, text = self.data_pairs[idx]\n"
      "        if isinstance(text, str) and \"||\" in text:   # [msc-mix] draw one prompt alternative per access\n"
      "            text = random.choice(text.split(\"||\")).strip()\n",
      "[msc-mix]")

# ---------------------------------------------------------------- 4) fewer checkpoint writes ----
patch("train.py",
      "        best_dice = mean_val_dice\n        torch.save({",
      "        best_dice = mean_val_dice\n        if os.environ.get(\"MSC_SAVE_BEST\") == \"1\": torch.save({   # [msc-save-best]",
      "[msc-save-best]")
patch("train.py",
      "    # Save the latest model\n    torch.save({",
      "    # Save the latest model\n    if (epoch + 1) % int(os.environ.get(\"MSC_SAVE_EVERY\", \"10\")) == 0 or epoch + 1 == num_epochs: torch.save({   # [msc-save-every]",
      "[msc-save-every]")

# ---------------------------------------------------------------- 3) raw stats in test.py ----
STATS = '''
# [msc-raw] ---- raw per-image uncertainty / logit / calibration statistics (added by msc_exp_patches.py) ----
_MSC_NB, _MSC_NH = 15, 50       # calibration bins on p = P(foreground); histogram bins on entropy (bits, 0..1)

def _msc_stats(logits, batch):
    """logits: raw MC logits [S, B, H, W] (any trailing shape). Returns one dict per image."""
    S, B = logits.shape[0], logits.shape[1]
    lg = logits.reshape(S, B, -1).float()
    pr = torch.sigmoid(lg)
    p = pr.mean(0)                                            # predictive mean          [B, P]
    ent = lambda q: -(q * torch.log(q + 1e-8) + (1 - q) * torch.log(1 - q + 1e-8)) / np.log(2)
    H_tot = ent(p)                                            # predictive entropy (bits)
    MI = (H_tot - ent(pr).mean(0)).clamp_min(0)               # mutual information = MC disagreement
    ml = lg.mean(0)                                           # mean logit
    sd = pr.std(0) if S > 1 else torch.zeros_like(p)
    pred = p > 0.5
    gt = batch.get("ground_truth_mask")
    gt = (gt.reshape(B, -1).to(p.device).float() > 0.5) if gt is not None and gt.numel() == p.numel() else None
    q = torch.tensor([0.01, 0.5, 0.99], device=p.device)
    rows = []
    for i in range(B):
        ql = torch.quantile(ml[i], q).tolist()
        r = {"Name": batch["mask_name"][i], "mc_n": S,
             "unc_mean": H_tot[i].mean().item(), "unc_max": H_tot[i].max().item(),
             "unc_in_pred": H_tot[i][pred[i]].mean().item() if pred[i].any() else float("nan"),
             "mi_mean": MI[i].mean().item(), "mc_sd_mean": sd[i].mean().item(), "p_max": p[i].max().item(),
             "logit_p01": ql[0], "logit_p50": ql[1], "logit_p99": ql[2],
             "frac_logit_gt17": (ml[i].abs() > 17).float().mean().item()}
        if gt is not None:
            g = gt[i]
            r["logit_bg_p50"] = ml[i][~g].median().item() if (~g).any() else float("nan")
            r["logit_fg_p50"] = ml[i][g].median().item() if g.any() else float("nan")
            b = (p[i] * _MSC_NB).long().clamp_(max=_MSC_NB - 1)
            r["_cal_n"] = torch.bincount(b, minlength=_MSC_NB).cpu().numpy()
            r["_cal_sp"] = torch.bincount(b, weights=p[i], minlength=_MSC_NB).cpu().numpy()
            r["_cal_pos"] = torch.bincount(b, weights=g.float(), minlength=_MSC_NB).cpu().numpy()
            h = (H_tot[i] * _MSC_NH).long().clamp_(0, _MSC_NH - 1)
            err = pred[i] != g
            r["_h_err"] = torch.bincount(h[err], minlength=_MSC_NH).cpu().numpy()
            r["_h_ok"] = torch.bincount(h[~err], minlength=_MSC_NH).cpu().numpy()
        rows.append(r)
    return rows

def _msc_write(rows, root, design):
    import pandas as pd
    keys = [k for k in ["_cal_n", "_cal_sp", "_cal_pos", "_h_err", "_h_ok"] if rows and all(k in r for r in rows)]
    arr = {k[1:]: np.stack([r[k] for r in rows]) for k in keys}
    pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows]).to_csv(
        os.path.join(root, f"raw_Prompt-{design}.csv"), index=False)
    if arr:
        np.savez_compressed(os.path.join(root, f"raw_Prompt-{design}.npz"),
                            names=np.array([r["Name"] for r in rows]), **arr)

def main():'''
patch("test.py", "\ndef main():", STATS, "[msc-raw]")
patch("test.py", "    with torch.no_grad():\n        for batch in tqdm(test_dataloader):\n",
      "    _msc_rows = []   # [msc-raw-rows]\n    with torch.no_grad():\n        for batch in tqdm(test_dataloader):\n",
      "[msc-raw-rows]")
patch("test.py", "            seg_samples = torch.sigmoid(seg_samples)\n",
      "            _msc_rows.extend(_msc_stats(seg_samples, batch))   # [msc-raw-collect]\n"
      "            seg_samples = torch.sigmoid(seg_samples)\n",
      "[msc-raw-collect]")
patch("test.py", "\nif __name__ == \"__main__\":",
      "    _msc_write(_msc_rows, results_root, cfg.prompt_design)   # [msc-raw-write]\n\nif __name__ == \"__main__\":",
      "[msc-raw-write]")

In [ ]:
!python /content/msc_exp_patches.py
!git diff --stat

In [ ]:
# Cache the UniMedCLIP backbone weights on Drive so they are downloaded only once
if USE_DRIVE:
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    if not os.path.islink("checkpoints"):
        !rm -rf checkpoints
        os.symlink(f"{DRIVE_DIR}/checkpoints", "checkpoints")
!ls -la checkpoints/ 2>/dev/null | head

### Data
Datasets come from the authors' Hugging Face repo (`TahaKoleilat/MedCLIPSeg`, one zip per dataset; zips are cached on Drive, so later sessions only unzip). The repo already contains the text prompts (`data/<DS>/Prompts_Folder`); the zips add `Train_Folder / Val_Folder / Test_Folder` with `img/` and `label/`.

Groups: **BUS** = BUSI → BUSBRA, BUSUC, BUID, UDIAT · **ENDO** = Kvasir → ColonDB, ClinicDB, CVC300, BKAI · **DERM** = ISIC → UWaterlooSkinCancer · **BRAIN** = BTMRI → BRISC.

In [ ]:
import shutil, zipfile, glob
from huggingface_hub import hf_hub_download

ZIP_DIR = f"{DRIVE_DIR}/zips" if USE_DRIVE else "/content/zips"

def get_dataset(name):
    dst = f"data/{name}"
    if glob.glob(f"{dst}/Test_Folder/img/*"):
        print(f"[have] {name}"); return
    zpath = hf_hub_download(repo_id="TahaKoleilat/MedCLIPSeg", repo_type="dataset",
                            filename=f"{name}.zip", local_dir=ZIP_DIR)
    tmp = f"/content/_unzip/{name}"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z: z.extractall(tmp)
    root = next(d for d, sub, _ in os.walk(tmp) if "Test_Folder" in sub or "Train_Folder" in sub)
    for item in os.listdir(root):
        src, out = os.path.join(root, item), os.path.join(dst, item)
        if item == "Prompts_Folder":            # keep the repo's prompt files, only add missing ones
            os.makedirs(out, exist_ok=True)
            for f in os.listdir(src):
                if not os.path.exists(os.path.join(out, f)): shutil.copy(os.path.join(src, f), out)
        elif os.path.isdir(src): shutil.copytree(src, out, dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"[ok]   {name}")

BUS = ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"]
for ds in BUS:
    get_dataset(ds)

In [ ]:
# Sanity check: every image listed in the prompt files exists, and count empty ground-truth masks
import pandas as pd, numpy as np, cv2
def check(name):
    rows = []
    for split, xl in [("Train", "Train_text.xlsx"), ("Val", "Val_text.xlsx"), ("Test", "Test_text_original.xlsx")]:
        p = f"data/{name}/Prompts_Folder/{xl}"
        if not os.path.exists(p): continue
        df = pd.read_excel(p)
        folder = f"data/{name}/{split}_Folder"
        missing = sum(not os.path.exists(f"{folder}/img/{i}") for i in df["Image"])
        empty = sum(cv2.imread(f"{folder}/label/{g}", 0).max() == 0 for g in df["Ground Truth"]
                    if os.path.exists(f"{folder}/label/{g}"))
        rows.append(dict(dataset=name, split=split, prompts=len(df), missing_imgs=missing, empty_masks=empty))
    return rows
display(pd.DataFrame(sum([check(d) for d in BUS], [])))

### Helpers: evaluate, persist, compare with the paper
`evaluate()` runs `test.py` + `utils/eval.py`, then **immediately** saves to Drive:
* a summary row → `results/all_results.csv` (reloaded automatically next session),
* the per‑image CSV, enriched with `gt_empty`, `pred_empty`, areas and the prompt text → `results/per_image/<run_id>.csv` (unique name per run, so nothing gets overwritten),
* masks + uncertainty maps → `results/maps/<run_id>.zip` (if `SAVE_MAPS`).

Summary columns: `IoU` = intersection over union (overlap ÷ union of predicted and true lesion), computed per image as DSC / (2 − DSC), which is exact for binary masks; `DSC_nonempty` = DSC over images whose ground truth has a lesion; `normals_pred_empty` = empty‑GT images where the model predicted nothing (each scores DSC = 1); `lesions_missed` = lesion images where the model predicted nothing.

In [ ]:
import re, time, datetime
RES_DIR = f"{DRIVE_DIR}/results" if USE_DRIVE else "/content/results"
PER_IMG_DIR, MAP_DIR, LOG_DIR, TAB_DIR = [f"{RES_DIR}/{d}" for d in ["per_image", "maps", "train_logs", "tables"]]
for d in [RES_DIR, PER_IMG_DIR, MAP_DIR, LOG_DIR, TAB_DIR]: os.makedirs(d, exist_ok=True)
ALL_CSV = f"{RES_DIR}/all_results.csv"
RESULTS = pd.read_csv(ALL_CSV).to_dict("records") if os.path.exists(ALL_CSV) else []
TABLES = {}          # analysis tables collected for save_tables()
print(f"{len(RESULTS)} result rows loaded from {ALL_CSV}")

PAPER = {  # MedCLIPSeg DSC (%) from the paper: Tables S14–S17 (data efficiency) and Table 2 (domain generalisation)
    ("BUSI",10):68.66, ("BUSI",25):77.73, ("BUSI",50):81.48, ("BUSI",100):85.72,
    ("BTMRI",10):79.07, ("BTMRI",25):83.93, ("BTMRI",50):85.93, ("BTMRI",100):88.03,
    ("ISIC",10):90.35, ("ISIC",25):91.00, ("ISIC",50):91.97, ("ISIC",100):92.54,
    ("Kvasir",10):77.21, ("Kvasir",25):84.21, ("Kvasir",50):88.18, ("Kvasir",100):90.15,
    ("Covid19",10):79.73, ("Covid19",25):81.83, ("Covid19",50):82.97, ("Covid19",100):83.41,
    ("BUSBRA",100):75.06, ("BUSUC",100):84.37, ("BUID",100):78.99, ("UDIAT",100):74.64,
    ("ColonDB",100):71.90, ("ClinicDB",100):80.80, ("CVC300",100):80.82, ("BKAI",100):79.15,
    ("BRISC",100):80.92, ("UWaterlooSkinCancer",100):83.53,
}
SUB = "MedCLIPSeg_unimedclip_ViT-B-16_Prompt-{}"
designs = ["original", "missing_location", "underdescriptive", "overdescriptive", "contradictory", "shuffled"]
NORMAL_RE = r"\bno (?:tumou?r|polyp|lesion)|\bnormal\b|unremarkable|healthy"
import matplotlib.pyplot as plt, textwrap

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-6000:]); raise RuntimeError("command failed")
    return p.stdout

def slug(s): return re.sub(r"[^A-Za-z0-9.]+", "-", str(s)).strip("-") or "default"
def run_id(target, source, pct, prompt, seed, opts, out):
    return f"{slug(out.replace('outputs_', ''))}__{source}-{target}__p{pct}__{prompt}__s{seed}__{slug(opts)}"

def prompts_for(target, prompt):
    return pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Description"].astype(str).str.strip()

def enrich(df, target, prompt, seg_dir=None):
    # add ground-truth / prediction facts per image
    lab = f"data/{target}/Test_Folder/label"
    gt = [cv2.imread(f"{lab}/{n}", 0) for n in df.Name]
    df["gt_empty"] = [g.max() == 0 for g in gt]
    df["gt_area"] = [float((g > 127).mean()) for g in gt]
    if seg_dir:
        pr = [cv2.imread(f"{seg_dir}/{n}", 0) for n in df.Name]
        df["pred_empty"] = [p.max() == 0 for p in pr]
        df["pred_area"] = [float((p > 127).mean()) for p in pr]
    else:   # legacy: only known for empty-GT images (DSC==1 <=> nothing predicted)
        df["pred_empty"] = np.where(df.gt_empty, df.DSC == 1, np.nan)
    try: df["prompt_text"] = df.Name.map(prompts_for(target, prompt))
    except FileNotFoundError: df["prompt_text"] = None
    return df

def iou(dsc):
    """Per-image IoU from per-image Dice: IoU = |A∩B| / |A∪B| = D / (2 - D) (exact; both-empty images: 1)."""
    return dsc / (2 - dsc)

def summarize(df, **meta):
    ge, pe = df.gt_empty.astype(bool), df.pred_empty
    r = dict(**meta, time=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), n=len(df),
             DSC=float(100 * df.DSC.mean()), NSD=float(100 * df.NSD.mean()),
             DSC_nonempty=float(100 * df.loc[~ge, "DSC"].mean()), NSD_nonempty=float(100 * df.loc[~ge, "NSD"].mean()),
             IoU=float(100 * iou(df.DSC).mean()), IoU_nonempty=float(100 * iou(df.loc[~ge, "DSC"]).mean()),
             n_empty=int(ge.sum()), normals_pred_empty=int((ge & (pe == True)).sum()),
             lesions_missed=int((~ge & (pe == True)).sum()) if pe.notna().all() else None)
    repro = meta["prompt"] == "original" and not meta.get("opts") and meta.get("out") in ("outputs_medclipseg", "outputs_train")
    r["paper_DSC"] = PAPER.get((meta["target"], meta["pct"])) if repro else None   # only genuine reproduction runs
    return r

def hist_auroc(pos, neg):
    """AUROC from two histograms over the same ordered bins: P(score_pos > score_neg) + 0.5 P(tie)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if pos.sum() == 0 or neg.sum() == 0: return float("nan")
    below = np.cumsum(neg) - neg
    return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

def calib_summary(z):
    """Pixel-level ECE (%, 15 bins on P(foreground)) and AUROC of predictive entropy for pixel errors."""
    n, sp, pos = z["cal_n"].sum(0), z["cal_sp"].sum(0), z["cal_pos"].sum(0)
    return dict(ECE=float(100 * np.abs(sp - pos).sum() / n.sum()), unc_AUROC=hist_auroc(z["h_err"].sum(0), z["h_ok"].sum(0)))

def evaluate(target, source=None, pct=100, prompt="original", out="outputs_medclipseg", opts="", tag="", seed=SEED, allow_cpu=False):
    source = source or target
    dev = ""
    if not HAS_GPU:
        if not allow_cpu: raise RuntimeError("No GPU in this runtime: evaluation would be very slow on CPU. Switch runtime, or pass allow_cpu=True.")
        dev = "MODEL.DEVICE cpu"
    t0 = time.time()
    sh(f"python test.py --config-file configs/{target}.yaml --output-dir {out} --source_dataset {source} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts} {dev}")
    t_test = round(time.time() - t0, 1)
    sh(f"python utils/eval.py --config-file configs/{target}.yaml --output-dir {out} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts}")
    secs = round(time.time() - t0, 1)
    name = target if pct == 100 else f"{target}_{pct}"
    seg_dir = f"{out}/{name}/seg_results/seed{seed}/{SUB.format(prompt)}"
    unc_dir = f"{out}/{name}/unc_results/seed{seed}/{SUB.format(prompt)}"
    df = enrich(pd.read_csv(f"{out}/{name}/seg_results/seed{seed}/test_Prompt-{prompt}.csv"), target, prompt, seg_dir)
    rid = run_id(target, source, pct, prompt, seed, opts, out)
    raw, extra = f"{out}/{name}/seg_results/seed{seed}/raw_Prompt-{prompt}.csv", {}
    if os.path.exists(raw) and os.path.getmtime(raw) >= t0:          # written by the v0.3 test.py patch
        df = df.merge(pd.read_csv(raw), on="Name", how="left")
        if df["unc_mean"].isna().all(): print("WARNING: raw stats matched no image names")
        if os.path.exists(raw[:-4] + ".npz"):
            os.makedirs(f"{RES_DIR}/raw", exist_ok=True); shutil.copy(raw[:-4] + ".npz", f"{RES_DIR}/raw/{rid}.npz")
            extra = calib_summary(np.load(raw[:-4] + ".npz"))
    df["IoU"] = iou(df.DSC)
    meta = dict(tag=tag, source=source, target=target, pct=pct, prompt=prompt, seed=seed, opts=opts, out=out, run_id=rid)
    for k, v in meta.items(): df[k] = v
    if SAVE_MAPS:
        with zipfile.ZipFile(f"{MAP_DIR}/{rid}.zip", "w") as z:
            for kind, d in [("mask", seg_dir), ("uncertainty", unc_dir)]:
                for f in os.listdir(d): z.write(f"{d}/{f}", f"{kind}/{f}")
    r = summarize(df, **meta); r["seconds"] = secs; r["seconds_test"] = t_test; r.update(extra)
    RESULTS.append(r); pd.DataFrame(RESULTS).to_csv(ALL_CSV, index=False)
    df.to_csv(f"{PER_IMG_DIR}/{rid}.csv", index=False)     # written last: run() treats this file as "evaluation done"
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items() if k not in ("run_id", "time", "out")})
    return df

def load_per_image(target, prompt="original", source=None, pct=100, seed=SEED, opts="", out="outputs_medclipseg"):
    source = source or ("BUSI" if target in BUS else target)
    p = f"{PER_IMG_DIR}/{run_id(target, source, pct, prompt, seed, opts, out)}.csv"
    if os.path.exists(p): return pd.read_csv(p)
    name = target if pct == 100 else f"{target}_{pct}"
    p = f"{PER_IMG_DIR}/legacy__{out}__{name}__s{seed}__{prompt}.csv"
    if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(f"no per-image results for {target}/{prompt} (pct={pct}, seed={seed}, out={out}) — run evaluate() first")

COLS = ["tag", "source", "target", "pct", "prompt", "seed", "opts", "n", "DSC", "IoU", "NSD", "DSC_nonempty", "IoU_nonempty",
        "n_empty", "normals_pred_empty", "lesions_missed", "paper_DSC", "delta", "ECE", "unc_AUROC", "seconds"]
def show_results(tag_prefix=None):
    d = pd.DataFrame(RESULTS)
    if d.empty: print("no results yet"); return
    d["delta"] = d.DSC - pd.to_numeric(d.get("paper_DSC"), errors="coerce")
    if tag_prefix: d = d[d.tag.astype(str).str.startswith(tag_prefix)]
    display(d[[c for c in COLS if c in d]].round(2))

def save_tables():
    d = pd.DataFrame(RESULTS)
    if len(d) and "run_id" in d:                           # re-runs append rows: keep the latest per run_id
        leg = d.run_id.astype(str).eq("legacy") | d.run_id.isna()
        d = pd.concat([d[leg], d[~leg].drop_duplicates("run_id", keep="last")])
    sheets = {"all_results": d}
    if len(d):
        d = d.assign(delta=d.DSC - pd.to_numeric(d.get("paper_DSC"), errors="coerce"))
        sheets["reproduction"] = d[d.paper_DSC.notna()][[c for c in COLS if c in d]]
        p = d[d.tag.astype(str).str.match(r"T2[ad]") & (d.opts.fillna("") == "")]
        if len(p):
            for v in ["DSC", "DSC_nonempty", "normals_pred_empty"]:
                sheets[f"prompts_{v}"] = p.pivot_table(index=["source", "target"], columns="prompt", values=v, aggfunc="last")
        m = d[d.tag.astype(str).str.startswith("T2b")]
        if len(m): sheets["mc_samples"] = m[[c for c in COLS if c in m]]
    sheets.update(TABLES)
    path = f"{TAB_DIR}/medclipseg_tables.xlsx"
    with pd.ExcelWriter(path) as xw:
        for name, t in sheets.items(): t.to_excel(xw, sheet_name=name[:31])
    for name, t in sheets.items(): t.to_csv(f"{TAB_DIR}/{name}.csv")
    print(f"saved {len(sheets)} tables -> {path}")

def make_shuffled(target, seed=0):
    # every image gets another test image's original prompt (deterministic for a given seed)
    p = f"data/{target}/Prompts_Folder/"
    df = pd.read_excel(p + "Test_text_original.xlsx")
    df["Description"] = df["Description"].sample(frac=1, random_state=seed).values
    df.to_excel(p + "Test_text_shuffled.xlsx", index=False)

for t in ["BUSI", "UDIAT"]: make_shuffled(t)

## R. Restore the first session's results (run once; safe to re‑run)
Session 1 saved `results_<time>.csv` (summary table) and `csv_logs_<time>.tgz` (per‑image CSVs + training logs) in `results/`. This imports both into the new format. Per‑image files from that session can only be partly enriched: predicted masks were not kept, so `pred_empty` is known only for empty‑GT images (DSC = 1 ⇔ nothing predicted). If an evaluation ran several times (the MC‑sample sweep), only its last version survived.

In [ ]:
done = {r.get("legacy_file") for r in RESULTS}
for f in sorted(glob.glob(f"{RES_DIR}/results_*.csv")):
    if os.path.basename(f) in done: print("[skip]", os.path.basename(f)); continue
    old = pd.read_csv(f)
    t2c = iter([1, 2])
    for _, row in old.iterrows():
        r = row.to_dict()
        r["seed"] = next(t2c) if str(r["tag"]).startswith("T2c") else SEED
        r["opts"] = "" if pd.isna(r.get("opts")) else r["opts"]
        r["out"] = "outputs_train" if str(r["tag"]).startswith("T3") else "outputs_medclipseg"
        n, ne = r["n"], r["n_empty"]
        r["normals_pred_empty"] = int(round((n * r["DSC"] - (n - ne) * r["DSC_nonempty"]) / 100)) if ne else 0
        r["legacy_file"] = os.path.basename(f); r["run_id"] = "legacy"
        RESULTS.append(r)
    print("[ok]  ", os.path.basename(f), len(old), "rows")
pd.DataFrame(RESULTS).to_csv(ALL_CSV, index=False)

for tgz in sorted(glob.glob(f"{RES_DIR}/csv_logs_*.tgz")):
    dst = f"/content/restored/{os.path.basename(tgz)[:-4]}"
    shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    subprocess.run(["tar", "-xzf", tgz, "-C", dst], check=True)
    for p in glob.glob(f"{dst}/**/*.csv", recursive=True) + glob.glob(f"{dst}/**/log.txt", recursive=True):
        rel = os.path.relpath(p, dst).split(os.sep)             # out/name/(seg_results|trained_models)/seedN/file
        out, name, kind, seed_dir, fname = rel[0], rel[1], rel[2], rel[3], rel[-1]
        target, s = re.sub(r"_\d+$", "", name), seed_dir.replace("seed", "")
        if fname == "log.txt":
            shutil.copy(p, f"{LOG_DIR}/legacy__{out}__{name}__s{s}__{kind}.txt"); continue
        prompt = fname[len("test_Prompt-"):-4]
        if not os.path.isdir(f"data/{target}/Test_Folder/label"): print("  (no labels for", target, "- skipped)"); continue
        df = enrich(pd.read_csv(p), target, prompt)
        df.to_csv(f"{PER_IMG_DIR}/legacy__{out}__{name}__s{s}__{prompt}.csv", index=False)
    print("[ok]  ", os.path.basename(tgz))
print(len(os.listdir(PER_IMG_DIR)), "per-image files,", len(os.listdir(LOG_DIR)), "training logs on Drive")
show_results()

## A. Data understanding (CPU is fine)
### A1. Prompt inventory
Every prompt file shipped with the repo: size, number of distinct prompts, length, how many look like "normal" prompts, and the most frequent prompt. Works for all 16 datasets because the prompt files are in the repo (no image download needed).

In [ ]:
DS_ALL = ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT", "Kvasir", "ColonDB", "ClinicDB", "CVC300", "BKAI",
          "ISIC", "UWaterlooSkinCancer", "BTMRI", "BRISC", "Covid19", "EUS"]
NORMAL_RE = r"\bno (?:tumou?r|polyp|lesion)|\bnormal\b|unremarkable|healthy"
rows = []
for ds in DS_ALL:
    for f in sorted(glob.glob(f"data/{ds}/Prompts_Folder/*.xlsx")):
        x = pd.read_excel(f)
        if "Description" not in x or x.empty: continue          # e.g. empty Val files of ColonDB / CVC300
        s = x["Description"].astype(str).str.strip(); vc = s.value_counts()
        rows.append(dict(dataset=ds, file=os.path.basename(f)[:-5], n=len(s), unique=s.nunique(),
                         mean_words=round(s.str.split().str.len().mean(), 1),
                         normal_like=int(s.str.contains(NORMAL_RE, case=False).sum()),
                         top_prompt=vc.index[0][:90], top_prompt_n=int(vc.iloc[0])))
inv = pd.DataFrame(rows); TABLES["A1_prompt_inventory"] = inv
pd.set_option("display.max_colwidth", 90)
display(inv[inv.dataset.isin(["BUSI", "UDIAT"])])

In [ ]:
# The same BUSI test image under every prompt design (one lesion image, one normal image)
designs = ["original", "missing_location", "underdescriptive", "overdescriptive", "contradictory", "shuffled"]
P = {d: prompts_for("BUSI", d) for d in designs}
o = P["original"]
for name in [o[~o.str.startswith("No tumor")].index[3], o[o.str.startswith("No tumor")].index[0]]:
    print(f"\n{name}"); [print(f"  {d:17s} {P[d][name]}") for d in designs]

### A2. Are the BUSI prompt "designs" controlled edits of the original?
The paper (Table 4) reads e.g. `missing_location` as "the original prompt without the location". This parses size, shape, class and location from each BUSI prompt and counts, per design, how often an attribute **changed** (both prompts state it, values differ) or was **dropped**. Only BUSI uses this template; the OOD sets use a different sentence style.

In [ ]:
def attrs(s):
    s = s.lower()
    return dict(size=tuple(re.findall(r"\b(small|medium|large)\b", s)),
                shape=tuple(re.findall(r"\b([\w]+)-shaped\b", s)),
                cls=tuple(re.findall(r"\b(benign|malignant|regular|irregular)\b", s)),
                loc=tuple(re.findall(r"\b(top left|top right|bottom left|bottom right|top|bottom|left|right|center)\b", s)))
rows = []
A0 = {k: attrs(v) for k, v in o.items()}
lesion = [k for k, a in A0.items() if a["shape"]]
for d in designs[1:-1]:
    Ad = {k: attrs(v) for k, v in P[d].items()}
    r = dict(design=d, lesion_prompts=len(lesion))
    for a in ["size", "shape", "cls", "loc"]:
        r[f"{a}_changed"] = sum(1 for k in lesion if Ad[k][a] and A0[k][a] and Ad[k][a] != A0[k][a])
        r[f"{a}_dropped"] = sum(1 for k in lesion if A0[k][a] and not Ad[k][a])
    rows.append(r)
TABLES["A2_busi_design_consistency"] = pd.DataFrame(rows)
display(TABLES["A2_busi_design_consistency"])

### A3. Do the prompts describe the ground‑truth mask?
For every image with a non‑empty mask: the stated location vs. the centroid of the largest mask component (3×3 grid; an axis the prompt doesn't mention counts as middle/center), stated lesion count vs. connected components, stated size vs. mask area. Also normal‑prompt ↔ empty‑mask agreement. High agreement means the text carries information taken from the annotation.

In [ ]:
from scipy import ndimage
NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
def stated_loc(s):
    s = s.lower().replace("ultrasound image", "")
    v = re.search(r"\b(top|upper|bottom|lower)\b", s); h = re.search(r"\b(left|right)\b", s)
    c = re.search(r"\b(cent(?:er|re|ral)|middle)\b", s)
    if not (v or h or c): return None
    return ("top" if v and v.group(1) in ("top", "upper") else "bottom" if v else "middle", h.group(1) if h else "center")
def mask_cell(m):
    lab, n = ndimage.label(m > 127)
    if n == 0: return None, 0
    sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))
    big = int(np.argmax(sizes)) + 1
    cy, cx = ndimage.center_of_mass(lab == big)
    H, W = m.shape
    v = "top" if cy < H / 3 else "bottom" if cy > 2 * H / 3 else "middle"
    h = "left" if cx < W / 3 else "right" if cx > 2 * W / 3 else "center"
    n_big = int((sizes >= 0.001 * m.size).sum())            # ignore specks < 0.1% of the image
    return (v, h), n_big

rows, per = [], []
for ds in BUS:
    for split, xl in [("Train", "Train_text.xlsx"), ("Val", "Val_text.xlsx"), ("Test", "Test_text_original.xlsx")]:
        p = f"data/{ds}/Prompts_Folder/{xl}"
        if not (os.path.exists(p) and os.path.isdir(f"data/{ds}/{split}_Folder/label")): continue
        df = pd.read_excel(p)
        for _, r in df.iterrows():
            m = cv2.imread(f"data/{ds}/{split}_Folder/label/{r['Ground Truth']}", 0)
            if m is None: continue
            s = str(r["Description"]).strip(); cell, ncomp = mask_cell(m)
            cnt = re.match(r"^(one|two|three|four|five)\b", s.lower())
            size = re.search(r"\b(small|medium|large)\b", s.lower())
            per.append(dict(dataset=ds, split=split, mask_empty=cell is None, normal_prompt=bool(re.search(NORMAL_RE, s, re.I)),
                            stated=stated_loc(s), mask_cell=cell, stated_count=NUM[cnt.group(1)] if cnt else None,
                            components=ncomp, stated_size=size.group(1) if size else None, area=float((m > 127).mean())))
per = pd.DataFrame(per)
for (ds, split), g in per.groupby(["dataset", "split"], sort=False):
    les = g[~g.mask_empty]; st = les[les.stated.notna()]
    one = st[(st.stated_count.isna()) | (st.stated_count == 1)]
    rows.append(dict(dataset=ds, split=split, images=len(g), empty_masks=int(g.mask_empty.sum()),
                     normal_prompt_and_empty=int((g.mask_empty & g.normal_prompt).sum()),
                     normal_prompt_but_lesion=int((~g.mask_empty & g.normal_prompt).sum()),
                     empty_mask_but_lesion_prompt=int((g.mask_empty & ~g.normal_prompt).sum()),
                     lesions_with_location=len(st),
                     loc_exact_3x3=round(100 * (one.stated == one.mask_cell).mean(), 1) if len(one) else None,
                     loc_vertical=round(100 * (one.stated.str[0] == one.mask_cell.str[0]).mean(), 1) if len(one) else None,
                     loc_horizontal=round(100 * (one.stated.str[1] == one.mask_cell.str[1]).mean(), 1) if len(one) else None,
                     count_matches=round(100 * (les.stated_count == les.components)[les.stated_count.notna()].mean(), 1) if les.stated_count.notna().any() else None))
TABLES["A3_prompt_vs_mask"] = pd.DataFrame(rows)
display(TABLES["A3_prompt_vs_mask"])
b = per[(per.dataset == "BUSI") & ~per.mask_empty & per.stated_size.notna()]
TABLES["A3_busi_size_vs_area"] = b.groupby("stated_size").area.describe()[["count", "25%", "50%", "75%"]].mul([1, 100, 100, 100]).round(2)
print("BUSI: mask area (% of image) by stated size"); display(TABLES["A3_busi_size_vs_area"])
x = per[(per.dataset == "BUSI") & ~per.mask_empty & per.stated.notna()]
TABLES["A3_busi_location_crosstab"] = pd.crosstab(x.stated.astype(str), x.mask_cell.astype(str))
print("BUSI: stated location (rows) vs mask-centroid cell (cols)"); display(TABLES["A3_busi_location_crosstab"])
save_tables()

### A4. Look at a few BUSI test images with their mask and prompt

In [ ]:
import matplotlib.pyplot as plt, textwrap
df = pd.read_excel("data/BUSI/Prompts_Folder/Test_text_original.xlsx").sample(8, random_state=1)
fig, ax = plt.subplots(2, 4, figsize=(16, 8.5))
for a, (_, r) in zip(ax.ravel(), df.iterrows()):
    img = cv2.cvtColor(cv2.imread(f"data/BUSI/Test_Folder/img/{r['Image']}"), cv2.COLOR_BGR2RGB)
    m = cv2.imread(f"data/BUSI/Test_Folder/label/{r['Ground Truth']}", 0)
    cs, _ = cv2.findContours((m > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(img, cs, -1, (255, 0, 0), 2)
    a.imshow(img); a.set_title("\n".join(textwrap.wrap(r["Description"], 45)), fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## B. Claim checks (CPU is fine; B3 is slow‑ish on CPU)
### B1. Normal BUSI test images under each prompt design
Uses the per‑image results (from Section R or from new runs). For an empty ground truth the DSC is binary: **1 = the model predicted nothing**, 0 = it predicted a lesion that isn't there.

In [ ]:
tabs = {}
for d in designs:
    try: tabs[d] = load_per_image("BUSI", d).set_index("Name")
    except FileNotFoundError as e: print("missing:", d)
normals = next(iter(tabs.values())).query("gt_empty").index
B1 = pd.DataFrame({d: t.loc[normals, "DSC"] for d, t in tabs.items()})
if "shuffled" in tabs:
    B1["prompt_given_when_shuffled"] = tabs["shuffled"].loc[normals, "prompt_text"]
    B1["shuffled_prompt_says_normal"] = B1.prompt_given_when_shuffled.str.contains(NORMAL_RE, case=False)
TABLES["B1_busi_normals_by_design"] = B1.reset_index()
display(B1)
print("normal images predicted empty, per design:", B1[[d for d in designs if d in B1]].sum().astype(int).to_dict())
if "shuffled" in B1:
    print(B1.groupby("shuffled_prompt_says_normal")["shuffled"].agg(["count", "sum"]).rename(columns={"sum": "predicted_empty"}))
save_tables()

### B2. The Dice loss in `train.py` is applied to `(B, H, W)` tensors
`calc_loss` passes logits and masks of shape `(B, 224, 224)` to MONAI's `DiceLoss`, which expects `(B, C, H, W)`. MONAI therefore treats the 224 image **rows as 224 classes**, drops row 0 (`include_background=False`) and computes a Dice per row. A row with no lesion only gets a low loss when the sum of its predicted probabilities is ≪ 1e‑5, i.e. background logits below about −17. Synthetic check with a **perfect** mask (no model needed):

In [ ]:
from monai.losses import DiceLoss
dl = DiceLoss(include_background=False, sigmoid=True, reduction="mean")      # exactly as in train.py
gt = torch.zeros(2, 224, 224); gt[0, 80:140, 90:150] = 1                       # image 0: lesion, image 1: normal
rows = []
for bg in [-2, -5, -10, -15, -17, -20, -25]:
    lg = torch.where(gt.bool(), torch.tensor(5.0), torch.tensor(float(bg)))   # perfect segmentation
    rows.append(dict(background_logit=bg, repo_loss_BHW=round(dl(lg, gt).item(), 3),
                     proper_loss_B1HW=round(dl(lg.unsqueeze(1), gt.unsqueeze(1)).item(), 3)))
TABLES["B2_dice_shape_demo"] = pd.DataFrame(rows); display(TABLES["B2_dice_shape_demo"])

### B3. Diagnostics on the released BUSI checkpoint
1. **Logits:** are background logits pushed below the −17 "cliff" the row‑wise Dice rewards? (Would explain the sudden drop of train *and* val loss around epoch 11 while val Dice rose smoothly.)
2. **Loss components** on a validation batch: repo Dice (row‑wise), proper Dice, BCE.
3. **Soft contrastive targets:** the code divides the text‑similarity logits by their row L2 norm before the softmax; paper Eq. 20 doesn't. How flat are the targets in each version? (1.0 normalised entropy = uniform.)

On CPU this takes a few minutes (model load + two batches).

In [ ]:
sys.path.insert(0, REPO)
from utils.main_utils import load_cfg_from_cfg_file, read_text
from trainers import build_medclipseg_unimedclip
from datasets.dataloader import DatasetSegmentation, ValGenerator
from monai.losses import DiceLoss
dev = "cuda" if HAS_GPU else "cpu"
sh(f"python utils/download_ckpt.py --dataset BUSI --seed {SEED}")
cfg = load_cfg_from_cfg_file("configs/BUSI.yaml"); cfg.MODEL.DEVICE = dev
model = build_medclipseg_unimedclip(cfg)
ck = torch.load(f"outputs_medclipseg/BUSI/trained_models/seed{SEED}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth", map_location=dev, weights_only=False)
model.load_state_dict(ck["model"]); model.to(dev).eval()

def batch(split, xl, n, n_normal=None, seed=0):
    ds = DatasetSegmentation(f"data/BUSI/{split}_Folder/", "BUSI", read_text(f"data/BUSI/Prompts_Folder/{xl}"), ValGenerator([224, 224]), image_size=224)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(ds))
    if n_normal is not None:
        nor = [i for i in idx if ds.data_pairs[i][2].startswith("No tumor")][:n_normal]
        idx = nor + [i for i in idx if not ds.data_pairs[i][2].startswith("No tumor")][:n - n_normal]
    items = [ds[int(i)] for i in idx[:n]]
    return (torch.stack([b["image"] for b in items]).to(dev), torch.stack([b["ground_truth_mask"] for b in items]).to(dev),
            [b["text_prompt"] for b in items])

imgs, masks, texts = batch("Val", "Val_text.xlsx", 12, n_normal=4)
with torch.no_grad():
    logits = model(image=imgs, text=texts, num_samples=1)[0]
m = masks.bool(); normal_img = ~m.flatten(1).any(1)
q = lambda x: np.percentile(x.float().cpu().numpy(), [1, 50, 99]).round(1).tolist()
B3a = pd.DataFrame([
    dict(pixels="background, lesion images", n=int((~m[~normal_img]).sum()), logit_p1_p50_p99=q(logits[~normal_img][~m[~normal_img]]), frac_below_minus17=float((logits[~normal_img][~m[~normal_img]] < -17).float().mean())),
    dict(pixels="background, normal images", n=int((~m[normal_img]).sum()), logit_p1_p50_p99=q(logits[normal_img]), frac_below_minus17=float((logits[normal_img] < -17).float().mean())),
    dict(pixels="lesion", n=int(m.sum()), logit_p1_p50_p99=q(logits[m]), frac_below_minus17=float((logits[m] < -17).float().mean())),
])
dl = DiceLoss(include_background=False, sigmoid=True); bce = torch.nn.BCEWithLogitsLoss()
B3b = pd.DataFrame([dict(model_logits=lab, repo_dice_BHW=dl(L, masks).item(), proper_dice_B1HW=DiceLoss(sigmoid=True)(L.unsqueeze(1), masks.unsqueeze(1)).item(),
                         bce=bce(L, masks.float()).item(), val_dice_at_0_5=(2 * ((L > 0) & m).sum() / ((L > 0).sum() + m.sum())).item())
                    for lab, L in [("as predicted", logits), ("same, logits clamped to >= -10", logits.clamp(min=-10))]]).round(3)
TABLES["B3a_logit_stats"], TABLES["B3b_loss_components"] = B3a, B3b
display(B3a); display(B3b)

In [ ]:
# Soft contrastive targets on a training-size batch (24) of training images + prompts
imgs, _, texts = batch("Train", "Train_text.xlsx", 24, seed=1)
with torch.no_grad():
    tok = model.tokenizer(texts).to(dev)
    emb = model.text_model.transformer.embeddings.word_embeddings(tok).type(model.dtype)
    _, txt = model.encode_text_image(tok, emb, imgs)
T = cfg.MODEL.TEMPERATURE
sim = txt @ txt.T / T
G_code = torch.softmax(sim / sim.norm(dim=-1, keepdim=True), -1)          # as in trainers/medclipseg_unimedclip.py
tn = txt / txt.norm(dim=-1, keepdim=True)
G_paper = torch.softmax(tn @ tn.T / T, -1)                                   # Eq. 20 with L2-normalised text features
def flat(G):
    H = -(G * G.clamp_min(1e-12).log()).sum(-1) / np.log(G.shape[1])
    return dict(max_prob=round(G.max(-1).values.mean().item(), 3), own_pair_prob=round(G.diag().mean().item(), 3),
                normalised_entropy=round(H.mean().item(), 3))
B3c = pd.DataFrame([dict(version="uniform (1/24)", max_prob=round(1/24, 3), own_pair_prob=round(1/24, 3), normalised_entropy=1.0),
                    dict(version="code (row-L2-normalised logits)", **flat(G_code)),
                    dict(version="paper Eq. 20 (normalised text)", **flat(G_paper))])
print("distinct prompts in this batch:", len(set(texts)), "of", len(texts))
TABLES["B3c_soft_targets"] = B3c; display(B3c)
save_tables()
del model; import gc; gc.collect()

## T1. Reproduce the paper from the released checkpoints (GPU)
`utils/download_ckpt.py` puts the authors' checkpoint where `test.py` looks. **Pass:** each DSC within ~±1 of the paper. (Already done in session 1 — Section R restored those rows; re‑run only if needed.)

In [ ]:
sh(f"python utils/download_ckpt.py --dataset BUSI --seed {SEED}")
evaluate("BUSI", tag="T1-ID")                                     # paper: 85.72
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:                  # paper: 75.06 / 84.37 / 78.99 / 74.64
    evaluate(tgt, source="BUSI", tag="T1-OOD")
for pct in [10, 25, 50]:                                          # paper: 68.66 / 77.73 / 81.48
    sh(f"python utils/download_ckpt.py --dataset BUSI --data_percentage {pct} --seed {SEED}")
    evaluate("BUSI", pct=pct, tag="T1-eff")
show_results("T1")

## T2. Probes on the released checkpoint (GPU)
### T2a. Prompt designs shipped with the repo + shuffled prompts

In [ ]:
for tgt in ["BUSI", "UDIAT"]:
    for design in designs:
        evaluate(tgt, source="BUSI", prompt=design, tag="T2a-prompt")
show_results("T2a")

### T2d. Controlled prompt edits (one attribute at a time) — BUSI test set
Because the repo's designs change several attributes at once (Section A2), these edits change exactly one thing in each lesion prompt and leave normal prompts untouched — except the last two, which test whether the model obeys the normal/lesion statement:

| edit | lesion prompts | normal prompts |
|---|---|---|
| `no_location` / `flip_location` | location removed / top↔bottom, left↔right | unchanged |
| `no_shape`, `no_size`, `no_class` | that attribute removed | unchanged |
| `generic` | "One tumor in the breast ultrasound image." (count kept) | unchanged |
| `lesion_as_normal` | "No tumor in the breast ultrasound image." | unchanged |
| `normal_as_lesion` | unchanged | "One tumor in the breast ultrasound image." |

In [ ]:
PAT = re.compile(r"^(One|Two|Three|Four|Five) (small|medium|large) ([\w-]+?)-shaped (benign|malignant|regular|irregular) (tumors?) at the (.+?) in the breast ultrasound image\.$")
NORMAL_TXT = "No tumor in the breast ultrasound image."
FLIP = {"top": "bottom", "bottom": "top", "left": "right", "right": "left"}
def edit(s, kind):
    s = str(s).strip()
    if s == NORMAL_TXT:
        return "One tumor in the breast ultrasound image." if kind == "normal_as_lesion" else s
    m = PAT.match(s)
    if not m: return s
    cnt, size, shape, cls, noun, loc = m.groups()
    if kind == "generic": return f"{cnt} {noun} in the breast ultrasound image."
    if kind == "lesion_as_normal": return NORMAL_TXT
    parts = [cnt, size, f"{shape}-shaped", cls, noun, f"at the {loc}"]
    if kind == "no_location": parts[5] = ""
    if kind == "flip_location": parts[5] = "at the " + re.sub(r"\b(top|bottom|left|right)\b", lambda w: FLIP[w.group(1)], loc)
    if kind == "no_shape": parts[2] = ""
    if kind == "no_size": parts[1] = ""
    if kind == "no_class": parts[3] = ""
    return " ".join(p for p in parts if p) + " in the breast ultrasound image."

base = pd.read_excel("data/BUSI/Prompts_Folder/Test_text_original.xlsx")
assert all(edit(s, "rebuild") == str(s).strip() for s in base.Description), "template parser does not reproduce the original prompts"
EDITS = ["no_location", "flip_location", "no_shape", "no_size", "no_class", "generic", "lesion_as_normal", "normal_as_lesion"]
for k in EDITS:
    e = base.copy(); e["Description"] = e.Description.map(lambda s: edit(s, k))
    e.to_excel(f"data/BUSI/Prompts_Folder/Test_text_{k}.xlsx", index=False)
print(base.Description.iloc[0]); [print(f"  {k:17s}", edit(base.Description.iloc[0], k)) for k in EDITS]

In [ ]:
for k in EDITS:
    evaluate("BUSI", prompt=k, tag="T2d-edit")
show_results("T2d")
save_tables()

### T2e. Follow-up prompt edits — size sweep, negation, which word restores detection (BUSI test, GPU ~18 min)
T2d showed that the size word sets mask extent and that "One tumor…" behaves like "No tumor…". These edits test that directly:

| edit | lesion prompts | normal prompts | question |
|---|---|---|---|
| `size_small` / `size_medium` / `size_large` | size word set to that value | unchanged | does the size word control mask extent? |
| `negated_full` | "No small oval-shaped … at the …" | unchanged | does the model read "No" in a full prompt? |
| `size_only` / `location_only` | "One small tumor in …" / "One tumor at the top in …" | unchanged | which word restores detection vs `generic`? |
| `normal_as_full_lesion` | unchanged | "One medium oval-shaped benign tumor at the center in …" | does a detailed prompt create a lesion? |

T2e‑3 needs the T2d per-image results (`original`, `no_size`, `generic`, `lesion_as_normal`, `normal_as_lesion`) on Drive; it reports anything missing.

In [ ]:
# ==== T2e-1. Follow-up prompt edits: build the prompt files (BUSI test set) ====
# Needs: Section 0 + the helpers cell (evaluate, load_per_image, show_results, save_tables).
import re
BUS = globals().get("BUS", ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"])   # load_per_image() needs it
P2E = "data/BUSI/Prompts_Folder/"
PAT = re.compile(r"^(One|Two|Three|Four|Five) (small|medium|large) ([\w-]+?)-shaped "
                 r"(benign|malignant|regular|irregular) (tumors?) at the (.+?) in the breast ultrasound image\.$")
NORMAL_TXT = "No tumor in the breast ultrasound image."
FULL_LESION_TXT = "One medium oval-shaped benign tumor at the center in the breast ultrasound image."
END = " in the breast ultrasound image."

def edit2(s, kind):
    s = str(s).strip()
    if s == NORMAL_TXT:                                   # normal images: only one edit touches them
        return FULL_LESION_TXT if kind == "normal_as_full_lesion" else s
    m = PAT.match(s)
    if m is None:                                         # fail loudly instead of silently keeping the prompt
        raise ValueError(f"unparsed prompt: {s!r}")
    cnt, size, shape, cls, noun, loc = m.groups()
    if kind.startswith("size_") and kind != "size_only":
        size = kind[len("size_"):]                        # size_small / size_medium / size_large
    elif kind == "negated_full":  cnt = "No"              # "No small oval-shaped ... tumor at ..."
    elif kind == "size_only":     return f"{cnt} {size} {noun}{END}"
    elif kind == "location_only": return f"{cnt} {noun} at the {loc}{END}"
    elif kind not in ("rebuild", "normal_as_full_lesion"):
        raise ValueError(kind)
    return f"{cnt} {size} {shape}-shaped {cls} {noun} at the {loc}{END}"

EDITS2 = ["size_small", "size_medium", "size_large",          # causal test: does the size word set mask extent?
          "negated_full",                                     # negation inside the full template
          "size_only", "location_only",                       # which attribute restores detection vs. generic
          "normal_as_full_lesion"]                            # normals get a detailed lesion prompt (controlled B1)

base2 = pd.read_excel(P2E + "Test_text_original.xlsx")
assert all(edit2(s, "rebuild") == str(s).strip() for s in base2.Description), "parser does not reproduce the originals"
for k in EDITS2:
    e = base2.copy(); e["Description"] = e.Description.map(lambda s: edit2(s, k))
    n_changed = (e.Description != base2.Description.str.strip()).sum()
    e.to_excel(P2E + f"Test_text_{k}.xlsx", index=False)
    print(f"{k:22s} {n_changed:2d}/78 prompts changed | e.g. {e.Description.iloc[0] if k != 'normal_as_full_lesion' else e.Description.iloc[1]}")

In [ ]:
# ==== T2e-2. Run them (GPU, ~2.5 min each, ~18 min total) ====
for k in EDITS2:
    evaluate("BUSI", prompt=k, tag="T2e-edit")
show_results("T2e")
save_tables()

In [ ]:
# ==== T2e-3. Analysis (CPU is fine; reads the per-image CSVs on Drive) ====
def _pi(prompt):
    try: return load_per_image("BUSI", prompt).set_index("Name")
    except FileNotFoundError: print(f"  (missing per-image results for '{prompt}' - run evaluate() for it)"); return None

orig = _pi("original")
lesions = orig.index[~orig.gt_empty.astype(bool)]
normals = orig.index[orig.gt_empty.astype(bool)]
stated = orig.loc[lesions, "prompt_text"].astype(str).str.extract(r"^\w+ (small|medium|large) ")[0]
SIZES = ["small", "medium", "large"]

# (a) Size sweep: same image, only the size word changes -------------------------------------------
size_conds = {"original": "original", "none": "no_size", "small": "size_small", "medium": "size_medium", "large": "size_large"}
st = {lab: t for lab, p in size_conds.items() if (t := _pi(p)) is not None}
ratio = pd.DataFrame({lab: t.loc[lesions, "pred_area"] / t.loc[lesions, "gt_area"] for lab, t in st.items()})
dsc = pd.DataFrame({lab: 100 * t.loc[lesions, "DSC"] for lab, t in st.items()})
ratio["true size"] = dsc["true size"] = stated
T2e_ratio = ratio.groupby("true size").median().reindex(SIZES).round(2)
T2e_ratio.insert(0, "n", stated.value_counts().reindex(SIZES))
T2e_dsc = dsc.groupby("true size").mean().reindex(SIZES).round(1)
print("Median predicted/true lesion area - rows: size word in the ORIGINAL prompt, cols: size word GIVEN"); display(T2e_ratio)
print("Mean lesion DSC (%)"); display(T2e_dsc)
if all(s in st for s in SIZES):
    a = pd.DataFrame({s: st[s].loc[lesions, "pred_area"] for s in SIZES})
    mono = ((a.small < a.medium) & (a.medium < a.large)).mean()
    print(f"Lesions whose predicted area grows small < medium < large: {100 * mono:.0f}%  "
          f"(median area ×{(a.large / a.small.clip(lower=1e-6)).median():.2f} from 'small' to 'large')")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for s in SIZES:
        ax.plot(SIZES, T2e_ratio.loc[s, SIZES], marker="o", label=f"true size: {s}")
    ax.axhline(1, color="gray", lw=0.8, ls="--"); ax.set_yscale("log")
    ax.set_xlabel("size word given in the prompt"); ax.set_ylabel("median predicted / true area")
    ax.set_title("Does the size word set the mask extent?", fontsize=10); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# (b) Presence: which wording makes lesions disappear? --------------------------------------------
pres_conds = ["original", "negated_full", "location_only", "size_only", "generic", "lesion_as_normal"]
pt = {p: t for p in pres_conds if (t := _pi(p)) is not None}
missed = {p: set(t.loc[lesions].query("pred_empty == True").index) for p, t in pt.items()}
T2e_presence = pd.DataFrame({p: {"example prompt": t.loc[lesions[0], "prompt_text"],
                                 "lesion DSC": round(100 * t.loc[lesions, "DSC"].mean(), 1),
                                 "lesions missed": len(missed[p]),
                                 "median pred/true area": round((t.loc[lesions, "pred_area"] / t.loc[lesions, "gt_area"]).median(), 2)}
                             for p, t in pt.items()}).T
print("Lesion images (65) under presence-related edits"); display(T2e_presence)
T2e_overlap = pd.DataFrame({a: {b: len(missed[a] & missed[b]) for b in missed} for a in missed})
print("Missed-lesion overlap (cell = number of lesions missed under BOTH; diagonal = missed under that edit)"); display(T2e_overlap)

# (c) Normals: does a detailed lesion prompt create a lesion? ---------------------------------------
norm_conds = ["original", "normal_as_lesion", "normal_as_full_lesion"]
nt = {p: t for p in norm_conds if (t := _pi(p)) is not None}
T2e_normals = pd.DataFrame({p: {"prompt given to normals": t.loc[normals[0], "prompt_text"],
                                "normals with a false positive": int((t.loc[normals, "pred_empty"] != True).sum()),
                                "mean predicted area (% image)": round(100 * t.loc[normals, "pred_area"].mean(), 2)}
                            for p, t in nt.items()}).T
print("Normal images (13)"); display(T2e_normals)

for name, t in {"T2e_size_area_ratio": T2e_ratio, "T2e_size_dsc": T2e_dsc, "T2e_presence": T2e_presence,
                "T2e_missed_overlap": T2e_overlap, "T2e_normals": T2e_normals}.items():
    TABLES[name] = t
save_tables()

### T2b. Monte‑Carlo samples vs. accuracy and runtime

In [ ]:
for k in [1, 5, 10, 30]:
    evaluate("BUSI", opts=f"TEST.NUM_SAMPLES {k}", tag=f"T2b-mc{k}")
show_results("T2b")

### T2c. Test‑time sampling noise (same checkpoint, different `--seed`)

In [ ]:
src = f"outputs_medclipseg/BUSI/trained_models/seed{SEED}"
for s in [1, 2]:
    os.makedirs(f"outputs_medclipseg/BUSI/trained_models/seed{s}", exist_ok=True)
    shutil.copy(f"{src}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth", f"outputs_medclipseg/BUSI/trained_models/seed{s}/")
    evaluate("BUSI", seed=s, tag="T2c-mcnoise")
show_results("T2c")

### Look at predictions
`show_cases()` shows the worst and best images of any evaluation (read from the maps zip on Drive, so it also works for earlier sessions when `SAVE_MAPS` was on).

In [ ]:
def show_cases(target, prompt="original", source=None, pct=100, seed=SEED, opts="", out="outputs_medclipseg", n=4):
    source = source or ("BUSI" if target in BUS else target)
    rid = run_id(target, source, pct, prompt, seed, opts, out)
    if not (os.path.exists(f"{PER_IMG_DIR}/{rid}.csv") and os.path.exists(f"{MAP_DIR}/{rid}.zip")):
        print(f"no saved masks for run {rid} - run evaluate() for it in this or a later session (with SAVE_MAPS=True)"); return
    df = pd.read_csv(f"{PER_IMG_DIR}/{rid}.csv").sort_values("DSC")
    imgs = pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Image"]
    z = zipfile.ZipFile(f"{MAP_DIR}/{rid}.zip")
    rd = lambda p, flag: cv2.imdecode(np.frombuffer(z.read(p), np.uint8), flag)
    picks = list(df.head(n // 2).Name) + list(df.tail(n - n // 2).Name)
    fig, ax = plt.subplots(len(picks), 4, figsize=(13, 3.2 * len(picks)))
    for r, m in enumerate(picks):
        row = df.set_index("Name").loc[m]
        img = cv2.cvtColor(cv2.imread(f"data/{target}/Test_Folder/img/{imgs[m]}"), cv2.COLOR_BGR2RGB)
        panels = [(img, textwrap.shorten(str(row.prompt_text), 60)), (cv2.imread(f"data/{target}/Test_Folder/label/{m}", 0), "ground truth"),
                  (rd(f"mask/{m}", 0), f"prediction  DSC={row.DSC:.2f}"), (cv2.cvtColor(rd(f"uncertainty/{m}", 1), cv2.COLOR_BGR2RGB), "uncertainty (per-image scaled)")]
        for c, (im, t) in enumerate(panels):
            ax[r, c].imshow(im, cmap="gray"); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

show_cases("BUSI")

## T3. Train from scratch (GPU)
Paper settings: batch 24, Adam 3e‑4, cosine schedule, 100 epochs, last‑epoch checkpoint. `train()` copies the training log to Drive (and the checkpoint if `KEEP_CKPT`). If Colab disconnects, re‑run setup and call `train(..., resume=True)`.

In [ ]:
OUT = "outputs_train"
def train(ds, pct=100, out=OUT, seed=SEED, opts="", resume=False, allow_cpu=False, env="", keep=None):
    if not HAS_GPU and not allow_cpu: raise RuntimeError("No GPU in this runtime: training needs a GPU.")
    dev = "" if HAS_GPU else "MODEL.DEVICE cpu"
    t0 = time.time()
    get_ipython().system(f"{env} python train.py --config-file configs/{ds}.yaml --output-dir {out} --data_percentage {pct} --seed {seed} {'--resume' if resume else ''} {opts} {dev}")
    name = ds if pct == 100 else f"{ds}_{pct}"
    d = f"{out}/{name}/trained_models/seed{seed}"
    tag = f"{slug(out.replace('outputs_', ''))}__{name}__s{seed}__{slug(opts)}"
    if os.path.exists(f"{d}/log.txt"): shutil.copy(f"{d}/log.txt", f"{LOG_DIR}/{tag}.txt")
    ck = f"{d}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth"
    assert os.path.exists(ck) and os.path.getmtime(ck) > t0, "training failed (no new checkpoint) - see the output above"
    n_ep = int(re.search(r"NUM_EPOCHS:\s*(\d+)", open(f"configs/{ds}.yaml").read()).group(1))
    m = re.search(r"TRAIN\.NUM_EPOCHS (\d+)", opts); n_ep = int(m.group(1)) if m else n_ep
    done_ep = max([int(x) for x in re.findall(r"EPOCH: (\d+) \|", open(f"{d}/log.txt").read())] or [0])
    assert done_ep == n_ep, f"training stopped at epoch {done_ep}/{n_ep} - see the output above; re-run to resume"
    import torch
    ck_ep = torch.load(ck, map_location="cpu", weights_only=False, mmap=True).get("epoch", -1) + 1
    assert ck_ep == n_ep, f"checkpoint holds epoch {ck_ep}/{n_ep} - training did not finish; re-run to resume"
    mins, TT = (time.time() - t0) / 60, f"{RES_DIR}/train_times.csv"
    pd.DataFrame([dict(tag=tag, pct=pct, seed=seed, minutes=round(mins, 1), resumed=bool(resume),
                       gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                       time=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))]).to_csv(TT, mode="a", header=not os.path.exists(TT), index=False)
    if (KEEP_CKPT if keep is None else keep):
        try: shutil.copy(ck, f"{RES_DIR}/{tag}_latest.pth"); print(f"checkpoint kept on Drive: {tag}_latest.pth")
        except OSError as err:   # e.g. Drive full: the training itself is fine, only the backup copy is missing
            print(f"WARNING: could not copy the checkpoint to Drive ({err}). Training is complete; "
                  "free Drive space and copy it manually if you need it later.")
    print(f"training took {mins:.1f} min; log saved as train_logs/{tag}.txt; time logged to train_times.csv")

def plot_log(path):
    log = open(path).read()
    ep = [(int(a), float(b), float(c)) for a, b, c in re.findall(r"EPOCH: (\d+) \| Training Loss: ([\d.]+) \| Validation Loss: ([\d.]+)", log)]
    dice = [float(x) for x in re.findall(r"(?:Dice: |→ )([\d.]+)\n", log)]
    e, tr, va = zip(*ep)
    plt.figure(figsize=(10, 3)); plt.subplot(1, 2, 1); plt.plot(e, tr, label="train"); plt.plot(e, va, label="val"); plt.legend(); plt.title("loss")
    plt.subplot(1, 2, 2); plt.plot(range(1, len(dice) + 1), dice); plt.title("val Dice (1 MC sample)"); plt.suptitle(os.path.basename(path), fontsize=8); plt.show()

print("\n".join(sorted(os.listdir(LOG_DIR))))

In [ ]:
train("BUSI", pct=10)
evaluate("BUSI", pct=10, out=OUT, tag="T3-train")
train("BUSI")
evaluate("BUSI", out=OUT, tag="T3-train")
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:
    evaluate(tgt, source="BUSI", out=OUT, tag="T3-train")
show_results("T3")
plot_log(f"{LOG_DIR}/train__BUSI__s{SEED}__default.txt")
save_tables()

## X. Experiments (v0.3.4 — aligned with the TA's feedback)

**Every session:** Section 0 (setup, install, both patch cells, data) → helpers cell → the T3 cell that *defines* `train()` (not the one that runs T3) → **X1, X2, X3** → the next unfinished step. Every step is safe to re-run: finished trainings and evaluations are skipped (`per_image/<run>.csv` on Drive is the "done" flag), and checkpoints run with `keep=True` are restored from Drive. If you change a prompt design after it was evaluated, delete its `per_image/*__<design>__*.csv` files or it will be skipped.

**Metrics.** Dice (DSC) and IoU, per image, then averaged. IoU = overlap ÷ union of predicted and true lesion; Dice = 2 × overlap ÷ (predicted area + true area). For one image IoU = DSC / (2 − DSC) exactly, so IoU is always ≤ Dice (Dice 0.80 ⇔ IoU 0.67). "lesion_" columns average over the 65 test images that contain a lesion; `missed` = lesion images with an empty prediction; `normal_FP` = normal images where something was predicted.

**TA protocol for prompt robustness (Ehsan):** (1) evaluate the original model under prompt variations (synonyms, paraphrases, word order), (2) implement the improvement (H1), (3) compare both on the same BUSI test set and the same metrics. Prompt variations (`DESIGN_GROUPS` in X2):

| Group | Designs | What changes |
|---|---|---|
| reference | `original` | — |
| wording — all information, **never seen by H1** | `syn_noun` `syn_size` `syn_class` `syn_loc` `syn_all` | test-only synonyms (tumor→nodule, small→tiny, benign→noncancerous, top→upper, …) |
| | `para_plain` `heldout_para` | paraphrases ("This breast ultrasound image shows…", "Breast sonogram demonstrating…") |
| | `order_adj` `order_front` `order_scramble` | word order: adjectives reordered, location first, all words scrambled |
| less information (seen by H1 in training) | `no_size` `no_location` `imagelevel` `generic` | attributes removed |
| less information, unseen wording | `heldout_generic` | "One mass seen on this breast sonogram." |
| paper's designs | `missing_location` `underdescriptive` `overdescriptive` `contradictory` | MedCLIPSeg's own robustness prompts — **not controlled edits**: they also change size/class words, so read them as "different prompt", not "same information" |
| misleading / counterfactual | `lesion_as_normal` `normal_as_lesion` `negated_full` `normal_as_full_lesion` `shuffled` `size_small/medium/large` | wrong information |

H1 trains with **different** synonyms (tumor→lesion, small→little/medium→mid-sized/large→big), two other adjective orders and three other paraphrases; X1 asserts that no test-only word and no test sentence appears in H1's training prompts.

| Finding (round 1) | Tested by |
|---|---|
| F1 size word = binned mask area (prompts leak the mask) | **H0** arms A/B/C; H1 size sweep |
| F2 presence follows sentence form | Step 1 (`negated_full`, `normal_as_full_lesion`); H0 arm B; H1 (varied normal wording) |
| F3 brittle to wording | **H1** vs baseline on the table above |
| F5 Dice-loss shape bug → extreme logits | **H2** (batch-pooled Dice) |
| uncertainty misses prompt-induced errors | **H3** (no training) — bridge to Fernando |

| Step | What | Trainings |
|---|---|---|
| 1 | released ckpt: full prompt suite (28 designs, MC 10) + raw stats | 0 |
| 2 | H0 at 10 %: A, B, C × seeds 666/1/2 — **done** | 9 |
| 3 | H1 (full suite) + H2 (core suite) at 100 %, seed 666, + external sets | 2 |
| 4 | H0 A, B, C at 100 %, seed 666 (C = baseline for H1, full suite) | 3 |
| 5 | seeds 1, 2: C, H1, H2, then H0 A/B | ≤ 10 |

Timing: 10 % trainings took ~45 min before the checkpoint-saving fix; a full prompt suite is ~28 × 75 s ≈ 35 min per checkpoint. Time the first 100 % run (`train()` prints it). Colab Pro needs the tab open.

**Pre-registered predictions** (printed by the analysis functions): H0 — C−B ≥ 3 lesion DSC and B≈A ⇒ leakage; B−A ≥ 3 ⇒ real text signal. H1 — generic prompt misses ≤ 5/65 lesions, original lesion DSC within 1.5 of baseline, size sweep stays monotonic; falsified if full-prompt DSC drops to arm A. H2 — median background logit > −25, lower ECE, higher MC mutual information and entropy-error AUROC, DSC within ±1.5. H3 — raw mean entropy detects missed lesions with AUROC < 0.65 on the released checkpoint. The TA does not require H1 to beat the original — a fair comparison and a clear explanation are the goal.

**Keep the test set honest:** every design choice in H1 (mixture, 45 % degraded, synonyms) was fixed before seeing H1 results. If you tune H1, tune on the validation split, not on this test suite, or say so in the notebook.

**Not in the main plan:** retraining BUSI 25 / 50 % and other dataset groups (evaluation-only, see the last cell).

In [ ]:
# ==== X1. Build every prompt file the experiments need (CPU, seconds). Re-run once per Colab session. ====
# Train/val prompts per arm go to their own folder (passed to train.py as DATASET.TEXT_PROMPT_PATH);
# test prompts go to the normal BUSI Prompts_Folder as Test_text_<design>.xlsx (used by evaluate(prompt=...)).
import zlib, random as _random
P0 = "data/BUSI/Prompts_Folder/"
IMG = "breast ultrasound image"
END = f" in the {IMG}."
NORMAL_TXT = f"No tumor in the {IMG}."
FULL_LESION_TXT = f"One medium oval-shaped benign tumor at the center in the {IMG}."
CONST_TXT = "A breast ultrasound image."
# permissive parser: train prompts also contain "One medium a fingerprint-shaped ..." and "Two small, medium ..."
PATX = re.compile(r"^(One|Two|Three|Four|Five) ((?:small|medium|large)(?:, (?:small|medium|large))*) (?:a )?([\w-]+?)-shaped "
                  r"(benign|malignant|regular|irregular) (tumors?) at the (.+?) in the breast ultrasound image\.$")

def parse(s):
    s = str(s).strip()
    if s == NORMAL_TXT: return None
    m = PATX.match(s)
    if m is None: raise ValueError(f"unparsed prompt: {s!r}")
    return dict(zip(["cnt", "size", "shape", "cls", "noun", "loc"], m.groups()))

def compose(d, size=True, shape=True, cls=True, loc=True, cnt=None, size_word=None):
    parts = [cnt or d["cnt"], (size_word or d["size"]) if size else "", f"{d['shape']}-shaped" if shape else "",
             d["cls"] if cls else "", d["noun"], f"at the {d['loc']}" if loc else ""]
    return " ".join(p for p in parts if p) + END

# ---- vocabulary: words used ONLY at test time (never in H1 training) vs synonyms H1 trains with ----
def sub_words(text, mapping):                       # whole-word replacement, keeps everything else
    return re.sub(r"\b(" + "|".join(map(re.escape, mapping)) + r")\b", lambda m: mapping[m.group(1)], text)
TEST_SYN = {   # test-only synonyms, one dictionary per prompt slot
    "noun":  {"tumor": "nodule", "tumors": "nodules"},
    "size":  {"small": "tiny", "medium": "moderate-sized", "large": "sizable"},
    "class": {"benign": "noncancerous", "malignant": "cancerous", "regular": "smooth-edged", "irregular": "rough-edged"},
    "loc":   {"top": "upper", "bottom": "lower", "center": "middle", "left": "left-hand side", "right": "right-hand side"},
}
TRAIN_SYN = {"noun": {"tumor": "lesion", "tumors": "lesions"},                 # H1 training synonyms (disjoint from TEST_SYN)
             "size": {"small": "little", "medium": "mid-sized", "large": "big"}}
TEST_ONLY_WORDS = ({w for m in TEST_SYN.values() for v in m.values() for w in v.split() if w != "side"} |
                   {"sonogram", "mass", "masses", "seen", "demonstrating", "shows"})

def syn(d, slots, image_phrase=False):
    """Original prompt with test-only synonyms in the given slots (same information, same word order)."""
    e = dict(d)
    if "noun" in slots:  e["noun"] = TEST_SYN["noun"][d["noun"]]
    if "size" in slots:  e["size"] = sub_words(d["size"], TEST_SYN["size"])
    if "class" in slots: e["cls"] = TEST_SYN["class"][d["cls"]]
    if "loc" in slots:   e["loc"] = sub_words(d["loc"], TEST_SYN["loc"])
    s = compose(e)
    return s.replace(END, " in the breast sonogram.") if image_phrase else s

def scramble(s):
    """All words in a random (but fixed per prompt) order: same words, no syntax."""
    w = s.rstrip(".").split(" ")
    rng = _random.Random(zlib.crc32(s.encode()))
    for _ in range(20):
        rng.shuffle(w)
        if " ".join(w) != s.rstrip("."): break
    return " ".join(w) + "."

# ---- test-time prompt designs ----
def test_prompt(s, kind):
    d = parse(s)
    if d is None:                                         # normal image
        return {"normal_as_lesion": "One tumor" + END, "normal_as_full_lesion": FULL_LESION_TXT, "constant": CONST_TXT,
                "heldout_para": "Breast sonogram demonstrating no mass.",
                "heldout_generic": "No mass seen on this breast sonogram.",
                "syn_noun": "No nodule" + END, "syn_all": "No nodule in the breast sonogram.",
                "para_plain": f"This {IMG} shows no tumor.",
                "order_front": f"In the {IMG}: no tumor.",
                "order_scramble": scramble(NORMAL_TXT)}.get(kind, NORMAL_TXT)   # syn_size/class/loc, order_adj: unchanged
    plural = d["cnt"] != "One"; mass = "masses" if plural else "mass"; cl = d["cnt"].lower()
    return {
        # information removed / changed (T2d, T2e)
        "generic":          compose(d, size=False, shape=False, cls=False, loc=False),
        "lesion_as_normal": NORMAL_TXT,
        "normal_as_lesion": str(s).strip(),
        "no_size":          compose(d, size=False),
        "no_location":      compose(d, loc=False),
        "size_small":       compose(d, size_word="small"),
        "size_medium":      compose(d, size_word="medium"),
        "size_large":       compose(d, size_word="large"),
        "negated_full":     compose(d, cnt="No"),
        "size_only":        compose(d, shape=False, cls=False, loc=False),
        "location_only":    compose(d, size=False, shape=False, cls=False),
        "normal_as_full_lesion": str(s).strip(),
        "constant":         CONST_TXT,                                               # H0 arm A
        "imagelevel":       compose(d, size=False, shape=False, loc=False),          # H0 arm B: presence, count, class
        # same information, different wording (TA: synonyms, paraphrases, word order) - all test-only for H1
        "syn_noun":         syn(d, ["noun"]),
        "syn_size":         syn(d, ["size"]),
        "syn_class":        syn(d, ["class"]),
        "syn_loc":          syn(d, ["loc"]),
        "syn_all":          syn(d, ["noun", "size", "class", "loc"], image_phrase=True),
        "para_plain":       f"This {IMG} shows {cl} {d['size']} {d['cls']} {d['noun']} with a {d['shape']} shape at the {d['loc']}.",
        "heldout_para":     f"Breast sonogram demonstrating {cl} {d['size']} {d['cls']} {mass} "
                            f"with a {d['shape']} shape, located at the {d['loc']}.",
        "order_adj":        f"{d['cnt']} {d['cls']} {d['shape']}-shaped {d['size']} {d['noun']} at the {d['loc']}{END}",  # class-shape-size
        "order_front":      f"In the {IMG}, at the {d['loc']}: {cl} {d['size']} {d['shape']}-shaped {d['cls']} {d['noun']}.",
        "order_scramble":   scramble(str(s).strip()),
        # same information reduced to presence + count, unseen wording
        "heldout_generic":  f"{d['cnt']} {mass} seen on this breast sonogram.",
    }[kind]

WORDING_DESIGNS = ["syn_noun", "syn_size", "syn_class", "syn_loc", "syn_all", "para_plain", "heldout_para",
                   "order_adj", "order_front", "order_scramble"]          # all-information, unseen wording
TEST_DESIGNS = ["generic", "lesion_as_normal", "normal_as_lesion", "no_size", "no_location", "size_small", "size_medium",
                "size_large", "negated_full", "size_only", "location_only", "normal_as_full_lesion",
                "constant", "imagelevel", "heldout_generic"] + WORDING_DESIGNS
base = pd.read_excel(P0 + "Test_text_original.xlsx")
for k in TEST_DESIGNS:
    e = base.copy(); e["Description"] = e.Description.map(lambda s: test_prompt(s, k))
    e.to_excel(P0 + f"Test_text_{k}.xlsx", index=False)
make_shuffled("BUSI")   # Test_text_shuffled.xlsx (defined in the helpers cell)

# ---- H1 training mixture: each cell holds 20 "||"-separated alternatives; the patched dataloader draws one per access ----
# 11 slots keep ALL the original information (original x4, 3 paraphrases, 2 training synonyms, 2 adjective orders),
# 9 slots carry LESS information (no size / no location / image-level / generic) -> 45 % degraded.
H1_NORMAL_ALTS = [f"No tumor is present in this {IMG}.",
                  f"Breast ultrasound image without any lesion.",
                  f"Normal {IMG}.",
                  f"Breast ultrasound image of normal breast tissue, with no lesion and no tumor anywhere in the image.",
                  f"No lesion{END}"]

def h1_alternatives(s, n=20):
    s = str(s).strip(); d = parse(s)
    if d is None:                                             # normals: half canonical, half varied (short AND long)
        alts = [NORMAL_TXT] * (n // 2) + [H1_NORMAL_ALTS[i % len(H1_NORMAL_ALTS)] for i in range(n - n // 2)]
        return "||".join(alts)
    cl = d["cnt"].lower(); plural = d["cnt"] != "One"
    les = "lesions" if plural else "lesion"; isare = "are" if plural else "is"
    tsyn = dict(d, noun=TRAIN_SYN["noun"][d["noun"]])
    full = [s, s, s, s,
            f"Breast ultrasound image with {cl} {d['size']} {d['cls']} {d['shape']}-shaped {d['noun']} at the {d['loc']}.",
            f"{d['cnt']} {d['noun']} ({d['size']}, {d['shape']}-shaped, {d['cls']}) located at the {d['loc']} of the {IMG}.",
            f"At the {d['loc']} of this {IMG} there {isare} {cl} {d['size']} {d['cls']} {les}.",
            compose(tsyn),                                                        # training synonym: noun
            compose(dict(d, size=sub_words(d["size"], TRAIN_SYN["size"]))),       # training synonym: size
            f"{d['cnt']} {d['shape']}-shaped {d['size']} {d['cls']} {d['noun']} at the {d['loc']}{END}",   # shape-size-class
            f"{d['cnt']} {d['cls']} {d['size']} {d['shape']}-shaped {d['noun']} at the {d['loc']}{END}"]   # class-size-shape
    deg = [compose(d, size=False), compose(d, loc=False),
           compose(d, size=False, shape=False, loc=False),                        # image-level
           compose(d, size=False, shape=False, cls=False, loc=False),             # generic
           compose(tsyn, size=False, shape=False, cls=False, loc=False),          # generic, training synonym
           f"Breast ultrasound image with {cl} {les}."]
    k_deg = n - len(full)
    return "||".join(full + [deg[i % len(deg)] for i in range(k_deg)])

ARMS = {"H0A": lambda s: CONST_TXT,
        "H0B": lambda s: test_prompt(s, "imagelevel") if parse(s) is not None else NORMAL_TXT,
        "H1mix": h1_alternatives}
for arm, fn in ARMS.items():
    os.makedirs(f"data/BUSI/Prompts_{arm}", exist_ok=True)
    for pct in [100, 10, 25, 50]:
        suf = "" if pct == 100 else f"_{pct}"
        for split in ["Train", "Val"]:
            df = pd.read_excel(P0 + f"{split}_text{suf}.xlsx")
            if not (arm == "H1mix" and split == "Val"):       # H1 validates on the original prompts
                df["Description"] = df.Description.map(fn)
            df.to_excel(f"data/BUSI/Prompts_{arm}/{split}_text{suf}.xlsx", index=False)

# ---- checks: the wording designs must be genuinely unseen by H1 ----
mix = pd.read_excel("data/BUSI/Prompts_H1mix/Train_text.xlsx").Description
train_alts = [a for cell in mix for a in cell.split("||")]
train_words = {w.strip(".,:()").lower() for a in train_alts for w in a.split()}
assert not (train_words & TEST_ONLY_WORDS), f"test-only words leaked into H1 training: {train_words & TEST_ONLY_WORDS}"
assert (mix.str.count(r"\|\|") == 19).all()
src = pd.read_excel(P0 + "Train_text.xlsx").Description.astype(str).str.strip()
for s0, cell in zip(src, mix):                                # same transformation applied to a TRAINING prompt
    alts = set(cell.split("||"))
    for k in WORDING_DESIGNS + ["heldout_generic"]:
        v = test_prompt(s0, k)
        if v == s0: continue                                  # design leaves this prompt unchanged (e.g. normals in syn_size)
        assert v not in alts, f"{k} would be seen in H1 training: {v!r}"
for k in WORDING_DESIGNS:                                     # wording designs keep every lesion attribute
    if k == "order_scramble": continue
    t = pd.read_excel(P0 + f"Test_text_{k}.xlsx").Description
    assert (t != base.Description.str.strip()).sum() >= 65, f"{k}: fewer prompts changed than expected"
ex_l = base.Description[~base.Description.str.strip().eq(NORMAL_TXT)].iloc[0]
print("lesion example:", ex_l.strip())
for k in ["constant", "imagelevel", "generic"] + WORDING_DESIGNS + ["heldout_generic"]: print(f"  {k:16s}", test_prompt(ex_l, k))
print("normal example:"); [print(f"  {k:16s}", test_prompt(NORMAL_TXT, k)) for k in ["syn_noun", "syn_all", "para_plain", "order_front", "order_scramble"]]
print("H1 alternatives (unique):"); [print("   ", a) for a in dict.fromkeys(h1_alternatives(ex_l).split("||"))]
print("H1 normal alternatives:", " | ".join(dict.fromkeys(h1_alternatives(NORMAL_TXT).split("||"))))
print("arm folders:", sorted(glob.glob("data/BUSI/Prompts_H*")))

In [ ]:
# ==== X2. Experiment runner (defines things only; nothing runs here) ====
# Guard: both patch cells (compatibility + experiment) must have run in THIS Colab VM — a new VM starts unpatched.
_need = {"utils/SurfaceDice.py": ("np.Inf", False), "test.py": ("[msc-raw]", True),
         "datasets/dataloader.py": ("[msc-mix]", True), "train.py": ("[msc-save-every]", True)}
_bad = [f for f, (pat, want) in _need.items() if (pat in open(f).read()) != want]
if _bad: raise RuntimeError(f"Not patched in this session: {_bad}. Run the two '%%writefile ..._patches.py' cells and the "
                            "two '!python /content/..._patches.py' cells in Section 0, then re-run this cell.")
if "WORDING_DESIGNS" not in globals(): raise RuntimeError("Run X1 (prompt files) before X2.")
SEEDS = [666, 1, 2]
KEEP_SEEDS = [666]   # checkpoints kept on Drive (~1 GB each): C, H1, H2 at these seeds -> 3 GB. Other seeds: a VM recycle mid-suite costs one retrain (~1 unit)
MC_FAST = "TEST.NUM_SAMPLES 10"     # robustness suites: MC 1/5/10/30 gave identical DSC (T2b), so 10 samples is enough there
EXP = {   # name -> training setup. Different `out` folders keep checkpoints and results apart.
    # keep=True: the checkpoint (~1 GB) is copied to Drive after training, so a recycled VM does not force a retrain
    # during the long prompt suites, and it can be re-evaluated later (new prompts, Fernando's validation split)
    "C":   dict(out="outputs_train", opts="", env="", own="original", keep=True),            # baseline = repo setup
    "H0A": dict(out="outputs_H0A", opts="DATASET.TEXT_PROMPT_PATH ./data/BUSI/Prompts_H0A/", env="", own="constant"),
    "H0B": dict(out="outputs_H0B", opts="DATASET.TEXT_PROMPT_PATH ./data/BUSI/Prompts_H0B/", env="", own="imagelevel"),
    "H1":  dict(out="outputs_H1mix", opts="DATASET.TEXT_PROMPT_PATH ./data/BUSI/Prompts_H1mix/", env="", own="original", keep=True),
    "H2":  dict(out="outputs_H2dice", opts="", env="MSC_DICE_FIX=1", own="original", keep=True),
    "REL": dict(out="outputs_medclipseg", opts=None, env="", own="original"),                # released checkpoint (no training)
}
DESIGN_GROUPS = {   # prompt variations evaluated on every baseline / H1 checkpoint (needs X1: WORDING_DESIGNS)
    "reference": ["original"],
    "wording, all information (unseen by H1)": WORDING_DESIGNS,
    "less information (seen by H1)": ["no_size", "no_location", "imagelevel", "generic"],
    "less information, unseen wording": ["heldout_generic"],
    "paper designs (not controlled: attributes change too)": ["missing_location", "underdescriptive", "overdescriptive", "contradictory"],
    "misleading / counterfactual": ["lesion_as_normal", "normal_as_lesion", "negated_full", "normal_as_full_lesion",
                                    "shuffled", "size_small", "size_medium", "size_large"],
}
ROBUST = [p for ps in DESIGN_GROUPS.values() for p in ps]                       # 28 designs, MC 10
CORE = ["original", "generic", "lesion_as_normal", "negated_full", "no_size", "shuffled",
        "underdescriptive", "heldout_generic"]                                   # smaller suite for H2 / H3
OOD = ["BUSBRA", "BUSUC", "BUID", "UDIAT"]

def ckpt_path(exp, pct, seed):
    name = "BUSI" if pct == 100 else f"BUSI_{pct}"
    return f"{EXP[exp]['out']}/{name}/trained_models/seed{seed}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth"

def ckpt_tag(exp, pct, seed):      # same naming as train() uses for logs / kept checkpoints on Drive
    e = EXP[exp]; name = "BUSI" if pct == 100 else f"BUSI_{pct}"
    return f"{slug(e['out'].replace('outputs_', ''))}__{name}__s{seed}__{slug(e['opts'])}"

def planned_evals(exp, pct, seed, evals=("own",), ood=False):
    e = EXP[exp]; todo = []
    for kind in evals:
        for p in {"own": [e["own"]], "robust": ROBUST, "core": CORE}[kind]:
            todo.append(("BUSI", p, "" if kind == "own" else MC_FAST))
    if ood and pct == 100: todo += [(t, "original", "") for t in OOD]
    return todo

def have_eval(exp, pct, seed, target, prompt, opts):
    rid = run_id(target, "BUSI", pct, prompt, seed, opts, EXP[exp]["out"])
    return os.path.exists(f"{PER_IMG_DIR}/{rid}.csv")

def restore_ckpt(exp, pct, seed):
    """Copy a kept checkpoint back from Drive (after the Colab VM was recycled)."""
    src, dst = f"{RES_DIR}/{ckpt_tag(exp, pct, seed)}_latest.pth", ckpt_path(exp, pct, seed)
    if os.path.exists(dst): return True
    if not os.path.exists(src): return False
    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst); open(dst + ".done", "w").close()
    print(f"[restored] {src}"); return True

def run(exp, pct=100, seed=SEED, evals=("own",), ood=False, keep=None):
    """Train (if needed) and evaluate. Safe to re-run: finished evaluations and finished training are skipped.
    keep=None -> keep the checkpoint on Drive if KEEP_CKPT is True, or the experiment is marked keep=True in EXP and seed is in KEEP_SEEDS."""
    e = EXP[exp]
    keep = (KEEP_CKPT or (e.get("keep", False) and seed in KEEP_SEEDS)) if keep is None else keep
    todo = [t for t in planned_evals(exp, pct, seed, evals, ood) if not have_eval(exp, pct, seed, *t)]
    if not todo: print(f"[done] {exp} pct={pct} seed={seed}"); return
    missing = sorted({p for t, p, _ in todo if not os.path.exists(f"data/{t}/Prompts_Folder/Test_text_{p}.xlsx")})
    if missing: raise RuntimeError(f"missing test prompt files {missing} - run X1 first (nothing was trained or evaluated)")
    ck = ckpt_path(exp, pct, seed)
    if exp == "REL":
        if not os.path.exists(ck): sh(f"python utils/download_ckpt.py --dataset BUSI --seed {seed}")
    elif not (os.path.exists(ck + ".done") or restore_ckpt(exp, pct, seed)):
        print(f"=== training {exp} pct={pct} seed={seed} (checkpoint {'kept on Drive' if keep else 'not kept'}) ===")
        if keep:
            try:
                free_gb = shutil.disk_usage(RES_DIR).free / 1e9
                if free_gb < 3: print(f"WARNING: only {free_gb:.1f} GB free on Drive - the ~1 GB checkpoint copy may fail")
            except OSError: pass
        train("BUSI", pct=pct, out=e["out"], seed=seed, opts=e["opts"], env=e["env"], keep=keep,
              resume=os.path.exists(ck))
        open(ck + ".done", "w").close()
    for target, prompt, opts in todo:
        evaluate(target, source="BUSI", pct=pct, prompt=prompt, out=e["out"], seed=seed, opts=opts,
                 tag=f"{exp}-{'robust' if opts else ('ood' if target != 'BUSI' else 'own')}")
    save_tables()

# ---- analysis helpers ----
def matched(a, b, prompt, pct=100, robust=True, target="BUSI"):
    """Restrict two groups of retrained checkpoints to the seeds evaluated in BOTH (paired comparison).
    A released-checkpoint baseline (REL) has no seed counterpart, so only group b is filtered then."""
    have = lambda ms: {s for x, s in ms if load_run(x, pct, s, prompt, robust, target) is not None}
    sb = have(b)
    if any(x == "REL" for x, _ in a): return a, [m for m in b if m[1] in sb]
    common = have(a) & sb
    return [m for m in a if m[1] in common], [m for m in b if m[1] in common]

def load_run(exp, pct, seed, prompt, robust=False, target="BUSI"):
    rid = run_id(target, "BUSI", pct, prompt, seed, MC_FAST if robust else "", EXP[exp]["out"])
    p = f"{PER_IMG_DIR}/{rid}.csv"
    return pd.read_csv(p) if os.path.exists(p) else None

def _truth(v): return v.map(lambda x: str(x).strip().lower() in ("true", "1", "1.0"))

def metrics(df):
    ge, pe = _truth(df.gt_empty), _truth(df.pred_empty)
    return dict(DSC=100 * df.DSC.mean(), IoU=100 * iou(df.DSC).mean(),
                lesion_DSC=100 * df.loc[~ge, "DSC"].mean(), lesion_IoU=100 * iou(df.loc[~ge, "DSC"]).mean(),
                missed=int((~ge & pe).sum()), normal_FP=int((ge & ~pe).sum()))

def group_metrics(members, prompt, pct=100, robust=False, target="BUSI"):
    """Average metrics over a list of (exp, seed) checkpoints; returns (mean dict, n_found)."""
    ms = [metrics(d) for d in (load_run(x, pct, s, prompt, robust, target) for x, s in members) if d is not None]
    return (pd.DataFrame(ms).mean().to_dict() if ms else {}), len(ms)

def paired_boot(members_a, members_b, prompt_a, prompt_b, pct=100, robust=False, n=2000, rng=0):
    """Lesion-image DSC difference B - A, averaged over seeds per image, with a paired bootstrap 95% CI."""
    def avg(members, prompt):
        ds = [d.set_index("Name") for d in (load_run(x, pct, s, prompt, robust) for x, s in members) if d is not None]
        if not ds: return None
        les = ~_truth(ds[0].gt_empty)
        return pd.concat([d.DSC for d in ds], axis=1).mean(axis=1)[les]
    a, b = avg(members_a, prompt_a), avg(members_b, prompt_b)
    if a is None or b is None: return None
    diff = 100 * (b - a.reindex(b.index)).values
    r = np.random.default_rng(rng)
    bs = [diff[r.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return dict(mean=diff.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), n_images=len(diff))

def calib_from_npz(paths):
    """Pooled pixel ECE (%) and pixel AUROC of entropy for errors, from the raw .npz files of some evaluations."""
    zs = [np.load(p) for p in paths if os.path.exists(p)]
    if not zs: return {}
    n, sp, pos = [sum(z[k].sum(0) for z in zs) for k in ["cal_n", "cal_sp", "cal_pos"]]
    he, ho = [sum(z[k].sum(0) for z in zs) for k in ["h_err", "h_ok"]]
    return dict(ECE=100 * np.abs(sp - pos).sum() / n.sum(), unc_AUROC=hist_auroc(he, ho))

In [ ]:
# ==== X3. Analysis functions (defines only). Each works with whatever has finished so far; missing runs are reported. ====
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

def baseline(robust=True):
    """Repo-trained baseline seeds (same pipeline as H1/H2); falls back to the released checkpoint until they exist."""
    c = [("C", s) for s in SEEDS if load_run("C", 100, s, "original", robust) is not None]
    if c: return c
    print("note: no retrained baseline (C) with this suite yet -> comparing against the released checkpoint (REL@666)")
    return [("REL", 666)]

def h0(pct):
    rows = []
    for arm, exp, prompt in [("A constant", "H0A", "constant"), ("B image-level", "H0B", "imagelevel"), ("C full", "C", "original")]:
        for s in SEEDS:
            d = load_run(exp, pct, s, prompt)
            if d is not None: rows.append(dict(arm=arm, seed=s, **metrics(d)))
    if not rows: print(f"H0 @{pct}%: no runs yet"); return
    t = pd.DataFrame(rows); TABLES[f"H0_runs_p{pct}"] = t
    g = t.groupby("arm")[["DSC", "IoU", "lesion_DSC", "lesion_IoU", "missed", "normal_FP"]].agg(["mean", "std", "count"]).round(2)
    TABLES[f"H0_summary_p{pct}"] = g; print(f"--- H0 @ BUSI {pct}% (train = test prompt type) ---"); display(g)
    m = t.groupby("arm").lesion_DSC.mean()
    if {"A constant", "B image-level", "C full"} <= set(m.index):
        cb, ba = m["C full"] - m["B image-level"], m["B image-level"] - m["A constant"]
        print(f"C-B = {cb:+.2f}  B-A = {ba:+.2f} lesion DSC (n seeds: {t.groupby('arm').size().to_dict()})")
        if cb >= 3 and abs(ba) < 3: print("=> gain is mostly LEAKAGE (mask-derived words), not language guidance")
        elif ba >= 3:               print("=> image-level text carries REAL signal worth keeping")
        else:                       print("=> inconclusive under the pre-registered 3-point rule; report effect sizes + CIs")
        pa = {"H0A": "constant", "H0B": "imagelevel", "C": "original"}
        for a, b in [("H0B", "C"), ("H0A", "H0B"), ("H0A", "C")]:
            ci = paired_boot([(a, s) for s in SEEDS], [(b, s) for s in SEEDS], pa[a], pa[b], pct=pct)
            if ci: print(f"   {b} - {a}: {ci['mean']:+.2f} [{ci['lo']:+.2f}, {ci['hi']:+.2f}] (paired bootstrap, {ci['n_images']} lesion images)")

def h1():
    """TA protocol: original model vs improvement, same test set, same metrics (Dice + IoU), per prompt variation."""
    H1S, base = [("H1", s) for s in SEEDS], baseline()
    rows = []
    for grp, ps in DESIGN_GROUPS.items():
        for p in ps:
            mb, mh = matched(base, H1S, p)
            b, nb = group_metrics(mb, p, robust=True); h, nh = group_metrics(mh, p, robust=True)
            if nb and nh:
                rows.append(dict(group=grp, prompt=p, n_base=nb, n_h1=nh,
                                 base_lesDSC=b["lesion_DSC"], h1_lesDSC=h["lesion_DSC"], d_lesDSC=h["lesion_DSC"] - b["lesion_DSC"],
                                 base_lesIoU=b["lesion_IoU"], h1_lesIoU=h["lesion_IoU"], d_lesIoU=h["lesion_IoU"] - b["lesion_IoU"],
                                 base_missed=b["missed"], h1_missed=h["missed"], base_normFP=b["normal_FP"], h1_normFP=h["normal_FP"]))
    if not rows: print("H1: need the robustness suite on a baseline AND at least one H1 checkpoint"); return
    t = pd.DataFrame(rows).set_index(["group", "prompt"]); TABLES["H1_robustness"] = t.round(2).reset_index(); display(t.round(2))
    print("n_base / n_h1 = seeds compared (matched: only seeds evaluated for BOTH models)")
    g = t.groupby(level="group", sort=False)[["base_lesDSC", "h1_lesDSC", "d_lesDSC", "base_lesIoU", "h1_lesIoU", "d_lesIoU",
                                               "base_missed", "h1_missed", "base_normFP", "h1_normFP"]].mean().round(2)
    TABLES["H1_by_group"] = g; print("--- mean per group of prompt variations ---"); display(g)
    ood = []
    for tgt in OOD:                                             # external datasets, original prompts
        mb, mh = matched(base, H1S, "original", robust=False, target=tgt)
        b, nb = group_metrics(mb, "original", target=tgt); h, nh = group_metrics(mh, "original", target=tgt)
        if nb and nh: ood.append(dict(target=tgt, seeds=nh, base_DSC=b["DSC"], h1_DSC=h["DSC"], base_IoU=b["IoU"], h1_IoU=h["IoU"]))
    if ood: o = pd.DataFrame(ood).set_index("target").round(2); TABLES["H1_ood"] = o; print("--- external datasets ---"); display(o)
    idx = lambda p: next((k for k in t.index if k[1] == p), None)
    print("Predictions:")
    if idx("generic"):  print(f"  generic: missed lesions {t.loc[idx('generic'),'base_missed']:.1f} -> {t.loc[idx('generic'),'h1_missed']:.1f} (target <= 5)")
    if idx("original"): print(f"  original: lesion DSC change {t.loc[idx('original'),'d_lesDSC']:+.2f} (target >= -1.5)")
    for p in ["generic", "syn_all", "order_scramble"]:
        ci = paired_boot(*matched(base, H1S, p), p, p, robust=True)
        if ci: print(f"  {p}: lesion DSC H1 - base {ci['mean']:+.2f} [{ci['lo']:+.2f}, {ci['hi']:+.2f}]")
    for name, members in [("base", base), ("H1", H1S)]:                   # does the model still READ the size word?
        fr, ratio = [], {}
        for x, s in members:
            ds = {z: load_run(x, 100, s, f"size_{z}", robust=True) for z in ["small", "medium", "large"]}
            if any(d is None for d in ds.values()): continue
            les = ~_truth(ds["small"].gt_empty)
            a = pd.concat([ds[z].set_index("Name").pred_area for z in ds], axis=1)[les.values]
            fr.append(((a.iloc[:, 0] < a.iloc[:, 1]) & (a.iloc[:, 1] < a.iloc[:, 2])).mean())
            for z in ds: ratio.setdefault(z, []).append((ds[z].pred_area / ds[z].gt_area)[les].median())
        if fr: print(f"  size sweep [{name}]: monotonic small<medium<large in {100*np.mean(fr):.0f}% of lesions; "
                     f"median pred/GT area {', '.join(f'{z} {np.mean(v):.2f}' for z, v in ratio.items())}")
    a, na = group_metrics([("H0A", s) for s in SEEDS], "constant")       # text-free arm A, 100 %
    c, nc = group_metrics([("C", s) for s in SEEDS], "original")        # repo prompts, 100 %
    h, nh = group_metrics(H1S, "original")
    if na and nc and nh:
        A, Cf, Hf = a["lesion_DSC"], c["lesion_DSC"], h["lesion_DSC"]
        print(f"  falsification check (full prompts, lesion DSC): text-free A {A:.2f} | repo C {Cf:.2f} | H1 {Hf:.2f}")
        if Cf - A < 1.5: print("   -> not testable: text adds < 1.5 DSC over the text-free arm (see H0)")
        else: print("   -> " + ("FALSIFIED: H1 sits closer to the text-free arm (robust because it ignores text)"
                                 if Hf - A < (Cf - A) / 2 else "ok: H1 keeps most of the text benefit"))

def raw_summary(members, prompt="original", robust=True):
    out = []
    for x, s in members:
        d = load_run(x, 100, s, prompt, robust)
        if d is None or "unc_mean" not in d: continue
        rid = run_id("BUSI", "BUSI", 100, prompt, s, MC_FAST if robust else "", EXP[x]["out"])
        out.append(dict(ckpt=f"{x}@{s}", **metrics(d), **calib_from_npz([f"{RES_DIR}/raw/{rid}.npz"]),
                        bg_logit_median=d.logit_bg_p50.median(), frac_abs_logit_gt17=d.frac_logit_gt17.mean(),
                        MC_mutual_info=d.mi_mean.mean(), MC_sd=d.mc_sd_mean.mean()))
    return pd.DataFrame(out)

def h2():
    t = pd.concat([raw_summary(baseline()).assign(group="repo loss"), raw_summary([("H2", s) for s in SEEDS]).assign(group="Dice fix")])
    if t.empty or "group" not in t or t.group.nunique() < 2: print("H2: need raw stats for the baseline and >= 1 Dice-fix checkpoint"); display(t); return
    TABLES["H2_calibration"] = t; display(t.round(3))
    g = t.groupby("group").mean(numeric_only=True)
    r, f = g.loc["repo loss"], g.loc["Dice fix"]
    print("Predictions (repo loss -> Dice fix):")
    print(f"  median background logit {r.bg_logit_median:.1f} -> {f.bg_logit_median:.1f} (target > -25)")
    print(f"  ECE {r.ECE:.2f} -> {f.ECE:.2f} (target: lower) | entropy AUROC for errors {r.unc_AUROC:.3f} -> {f.unc_AUROC:.3f} (target: higher)")
    print(f"  MC mutual information {r.MC_mutual_info:.2e} -> {f.MC_mutual_info:.2e} (target: higher)")
    print(f"  DSC {r.DSC:.2f} -> {f.DSC:.2f} | IoU {r.IoU:.2f} -> {f.IoU:.2f} (target: within 1.5 DSC)")

def h3(groups=None, prompts=tuple(CORE)):
    groups = groups or (("released", [("REL", 666)]), ("repo-trained C", [("C", s) for s in SEEDS]),
                        ("H1", [("H1", s) for s in SEEDS]), ("H2", [("H2", s) for s in SEEDS]))
    rows = []
    for gname, members in groups:
        parts = []
        for x, s in members:
            for p in prompts:
                d = load_run(x, 100, s, p, robust=True)
                if d is not None and "unc_mean" in d: parts.append(d.assign(prompt=p, ckpt=f"{x}@{s}"))
        if not parts: continue
        d = pd.concat(parts); d = d[~_truth(d.gt_empty)]
        y = _truth(d.pred_empty).astype(int)                          # 1 = lesion missed entirely
        r = dict(group=gname, lesion_cases=len(d), missed=int(y.sum()))
        for sc in ["unc_mean", "unc_max", "mi_mean"]:
            r[f"AUROC_{sc}"] = roc_auc_score(y, d[sc]) if 0 < y.sum() < len(y) else np.nan
        r["spearman_unc_vs_error"] = spearmanr(d.unc_mean, 1 - d.DSC).correlation
        rows.append(r)
    if not rows: print("H3: run Step 1 first"); return
    t = pd.DataFrame(rows).set_index("group").round(3); TABLES["H3_uncertainty"] = t; display(t)
    if "released" in t.index:
        a = t.loc["released", "AUROC_unc_mean"]
        if np.isnan(a): print("H3: the released checkpoint missed no lesions under these prompts -> AUROC undefined"); return
        print(f"Prediction: raw mean entropy detects prompt-induced misses with AUROC < 0.65 on the released ckpt -> {a:.3f} "
              f"({'SUPPORTED: misses are not flagged' if a < 0.65 else 'NOT supported'})"
              + ("; AUROC < 0.5 means missed lesions are MORE confident than detected ones" if a < 0.5 else ""))

def repro():
    """Training reproducibility of the repo setup across seeds (C) vs the paper (paper reports DSC only)."""
    rows = []
    for s in SEEDS:
        for t, pct in [("BUSI", 10), ("BUSI", 100)] + [(o, 100) for o in OOD]:
            d = load_run("C", pct, s, "original", target=t)
            if d is not None: rows.append(dict(target=t, pct=pct, seed=s, **metrics(d)))
    if not rows: print("no C runs found"); return
    t = pd.DataFrame(rows).groupby(["target", "pct"]).agg(DSC_mean=("DSC", "mean"), DSC_std=("DSC", "std"),
                                                            IoU_mean=("IoU", "mean"), seeds=("seed", "count"))
    t["paper_DSC"] = [PAPER.get(k) for k in t.index]; t["delta"] = t["DSC_mean"] - t["paper_DSC"]
    TABLES["repro_seeds"] = t.round(2); display(t.round(2))

### Step 1 — released checkpoint: full prompt suite + raw stats (GPU, no training)
Original-model evaluation under synonyms / paraphrases / word order (TA step 1); also the H3 baseline. Only designs not evaluated yet are run (~12 new ones, ≈15 min).

In [ ]:
run("REL", evals=("robust",))
_d = load_run("REL", 100, 666, "original", robust=True)          # 'original' MC10 was reused from T2b (no raw stats)
if _d is None or "unc_mean" not in _d:
    if not os.path.exists(ckpt_path("REL", 100, 666)): sh(f"python utils/download_ckpt.py --dataset BUSI --seed 666")
    evaluate("BUSI", prompt="original", opts=MC_FAST, tag="REL-robust")
_t = pd.DataFrame([dict(group=g, prompt=p, **group_metrics([("REL", 666)], p, robust=True)[0])
                   for g, ps in DESIGN_GROUPS.items() for p in ps]).set_index(["group", "prompt"]).round(2)
TABLES["REL_prompt_suite"] = _t.reset_index(); display(_t)        # TA step 1: original model under prompt variations
h3(groups=(("released", [("REL", 666)]),))

### Step 2 — H0 gate at 10 % data (GPU, short trainings)
Arms A (constant sentence), B (image-level: presence, count, class), C (repo prompts) × 3 seeds. (C seed 666 was retrained: the T3 result is stored in the old legacy format.)

In [ ]:
for s in SEEDS:
    for exp in ["H0A", "H0B", "C"]:
        run(exp, pct=10, seed=s)
h0(10)

### Step 3 — H1 (full suite) and H2 (core suite) at 100 %, seed 666 (GPU, 2 trainings; checkpoints kept on Drive for Fernando / H3)

In [ ]:
run("H1", seed=666, evals=("own", "robust"), ood=True, keep=True)
run("H2", seed=666, evals=("own", "core"), ood=True, keep=True)
h1(); h2(); h3()

### Step 4 — H0 at 100 %, seed 666: arms A, B, C (GPU, 3 trainings)

In [ ]:
run("H0A", seed=666)
run("H0B", seed=666)
run("C", seed=666, evals=("own", "robust"), ood=True)   # arm C (T3 result is legacy-format, so retrained once)
h0(100)

### Step 5 — extra seeds (GPU; in priority order — stop when the budget runs out)
`C` seeds 1, 2 double as the **training-reproducibility seeds** (`repro()`) and as baselines for H1/H2.

In [ ]:
for s in [666, 1, 2]:   # 666 normally already done in Step 4
    run("C", seed=s, evals=("own", "robust"), ood=True)
for s in [1, 2]:
    run("H1", seed=s, evals=("own", "robust"), ood=True)
    run("H2", seed=s, evals=("own", "core"))
for s in [1, 2]:
    run("H0A", seed=s)
    run("H0B", seed=s)

### All analyses (CPU is fine — run any time, e.g. on a CPU runtime after Section 0 + X1–X3)

In [ ]:
repro()
for pct in [10, 100]: h0(pct)
h1(); h2(); h3()
save_tables()

### Budget check — measured times → GPU-hours and units still needed (CPU is fine; run X1–X3 first)
Uses the median measured time per evaluation (from `all_results.csv`) and per training (from `train_times.csv`; an assumption is shown as ASSUMED until a training of that size has finished). Edit `UNITS_PER_HOUR` to the rate Colab shows.

In [ ]:
# ==== Budget check: measured times -> what is left and what it costs (CPU is fine; needs X1-X3) ====
UNITS_PER_HOUR = 1.2      # Colab compute units per GPU-hour - check the rate shown in Colab's "Resources" panel and edit
ASSUMED_TRAIN_MIN = {10: 45, 100: 60}   # used only until a training of that size has been timed in train_times.csv
PLAN = [  # mirrors the step cells: (step, exp, pct, seed, evals, ood)
    ("1", "REL", 100, 666, ("robust",), False),
    ("3", "H1", 100, 666, ("own", "robust"), True), ("3", "H2", 100, 666, ("own", "core"), True),
    ("4", "H0A", 100, 666, ("own",), False), ("4", "H0B", 100, 666, ("own",), False), ("4", "C", 100, 666, ("own", "robust"), True),
    ("5 baseline seeds", "C", 100, 1, ("own", "robust"), True), ("5 baseline seeds", "C", 100, 2, ("own", "robust"), True),
    ("5 H1/H2 seeds", "H1", 100, 1, ("own", "robust"), True), ("5 H1/H2 seeds", "H1", 100, 2, ("own", "robust"), True),
    ("5 H1/H2 seeds", "H2", 100, 1, ("own", "core"), False), ("5 H1/H2 seeds", "H2", 100, 2, ("own", "core"), False),
    ("5 H0 seeds", "H0A", 100, 1, ("own",), False), ("5 H0 seeds", "H0A", 100, 2, ("own",), False),
    ("5 H0 seeds", "H0B", 100, 1, ("own",), False), ("5 H0 seeds", "H0B", 100, 2, ("own",), False),
]

# --- measured evaluation times (seconds per evaluate() call), by dataset and MC setting ---
ev = pd.DataFrame(RESULTS)
ev = ev[pd.to_numeric(ev.get("seconds"), errors="coerce").notna()].assign(seconds=lambda d: d.seconds.astype(float),
                                                                           opts=lambda d: d.opts.fillna(""))
eval_sec = ev.groupby(["target", "opts"]).seconds.median().to_dict()
def eval_cost(target, opts):
    return eval_sec.get((target, opts), eval_sec.get(("BUSI", opts), 80.0))

# --- measured training times (minutes), by data percentage, from train_times.csv (written by train() since v0.3.2) ---
TT = f"{RES_DIR}/train_times.csv"
tt = pd.read_csv(TT) if os.path.exists(TT) else pd.DataFrame(columns=["pct", "minutes", "resumed"])
full = tt[tt.resumed.astype(str).str.lower() != "true"] if len(tt) else tt       # resumed runs are partial
train_min = {p: (float(full[full.pct == p].minutes.median()), "measured") if (full.pct == p).any() else (ASSUMED_TRAIN_MIN[p], "ASSUMED")
             for p in ASSUMED_TRAIN_MIN}

rows = []
for step, exp, pct, seed, evals, ood in PLAN:
    todo = [t for t in planned_evals(exp, pct, seed, evals, ood) if not have_eval(exp, pct, seed, *t)]
    trained = exp == "REL" or have_eval(exp, pct, seed, "BUSI", EXP[exp]["own"], "") or os.path.exists(ckpt_path(exp, pct, seed) + ".done")
    t_min = 0 if trained else train_min[pct][0]
    e_min = sum(eval_cost(t, o) for t, _, o in todo) / 60
    rows.append(dict(step=step, run=f"{exp}@{seed}", train_left=not trained, evals_left=len(todo), train_min=t_min, eval_min=round(e_min)))
b = pd.DataFrame(rows)
s = b.groupby("step", sort=False).agg(runs=("run", "count"), trainings_left=("train_left", "sum"), evals_left=("evals_left", "sum"),
                                      train_min=("train_min", "sum"), eval_min=("eval_min", "sum"))
s["hours"] = ((s.train_min + s.eval_min) / 60).round(1); s["units"] = (s.hours * UNITS_PER_HOUR).round(1)
display(s)
print(f"training time per run: 10% {train_min[10][0]:.0f} min ({train_min[10][1]}), 100% {train_min[100][0]:.0f} min ({train_min[100][1]}) | "
      f"eval: BUSI MC10 {eval_cost('BUSI', MC_FAST):.0f} s, BUSI MC30 {eval_cost('BUSI', ''):.0f} s per call")
print(f"TOTAL LEFT: {s.hours.sum():.1f} GPU-h ≈ {s.units.sum():.0f} units at {UNITS_PER_HOUR} units/h")
if train_min[100][1] == "measured":
    m = train_min[100][0]
    print(f"Estimate check: a 100% training took {m:.0f} min; the plan assumed 45-70 min -> "
          f"{'WITHIN the estimate' if 45 <= m <= 70 else ('FASTER than estimated' if m < 45 else 'SLOWER than estimated - re-plan')}")
if len(tt): print("\nmeasured trainings:"); display(tt.tail(10))
if "seconds_test" in ev:
    x = ev[ev.seconds_test.notna()]
    if len(x): print(f"evaluation split (median of {len(x)} calls): inference {x.seconds_test.median():.0f} s, "
                     f"scoring {(x.seconds - x.seconds_test).median():.0f} s per call")

## Save everything
`save_tables()` is also called at the end of each section. Files: `results/tables/medclipseg_tables.xlsx` (one sheet per table) and one CSV per table.

In [ ]:
save_tables()
!ls -la "{TAB_DIR}"

## Next: other dataset groups (GPU)
```python
for ds in ["Kvasir", "ColonDB", "ClinicDB", "CVC300", "BKAI"]: get_dataset(ds)
sh(f"python utils/download_ckpt.py --dataset Kvasir --seed {SEED}")
evaluate("Kvasir", tag="ENDO-ID")
for t in ["ColonDB", "ClinicDB", "CVC300", "BKAI"]: evaluate(t, source="Kvasir", tag="ENDO-OOD")
```
ISIC → UWaterlooSkinCancer and BTMRI → BRISC work the same way. BTMRI's test set has many normal scans with empty masks: watch `DSC_nonempty` and `normals_pred_empty`.